In [ ]:
# Purpose: Build the clean Pb(111) parent slab from the optimized bulk Pb calculation.
# Input: bulk/Pb_fcc/CONTCAR
# Output: structures/Pb111/clean/POSCAR
# Geometry: 3x3 surface cell, 4 Pb layers, 15 Å vacuum on each side.
# Constraints: bottom 2 Pb layers fixed; top 2 layers free.
# Assumptions: optimized bulk Pb remains cubic, so its three lattice-vector lengths are equal.

from pathlib import Path

import numpy as np
from ase.build import fcc111
from ase.constraints import FixAtoms
from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
BULK_PATH = ROOT / "bulk" / "Pb_fcc" / "CONTCAR"
OUT_DIR = ROOT / "structures" / "Pb111" / "clean"
OUT_PATH = OUT_DIR / "POSCAR"

if not BULK_PATH.is_file():
    raise FileNotFoundError(f"Missing optimized bulk structure: {BULK_PATH}")

bulk = read(BULK_PATH, format="vasp")
lengths = bulk.cell.lengths()

if not np.allclose(lengths, lengths[0], atol=1e-5):
    raise ValueError(f"Expected cubic optimized Pb bulk cell, got lengths: {lengths}")

a = float(lengths[0])

slab = fcc111(
    "Pb",
    size=(3, 3, 4),
    a=a,
    vacuum=15.0,
    orthogonal=False,
)

tags = slab.get_tags()
fixed_mask = tags >= 3
slab.set_constraint(FixAtoms(mask=fixed_mask))

OUT_DIR.mkdir(parents=True, exist_ok=True)

if OUT_PATH.exists():
    raise FileExistsError(f"Refusing to overwrite existing structure: {OUT_PATH}")

write(OUT_PATH, slab, format="vasp", direct=True, vasp5=True)

fixed = np.flatnonzero(fixed_mask).tolist()

print(f"Bulk lattice constant: {a:.8f} Å")
print(f"Atoms: {len(slab)}")
print(f"PBC: {slab.pbc.tolist()}")
print(f"Layer tags: {sorted(set(tags))}")
print(f"Fixed atoms: {len(fixed)} / {len(slab)}")
print(f"Fixed indices: {fixed}")
print("Cell:")
print(slab.cell)
print(f"Wrote: {OUT_PATH}")

In [ ]:
# Purpose: Generate the first parallel batch of Pb(111) surface calculations.
#
# Inputs:
#   bulk/Pb_fcc/CONTCAR
#   structures/Pb111/clean/POSCAR
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   S01-S04: clean Pb(111) parent and numerical convergence calculations
#   H01-H04: one-H adsorption-site screening calculations
#   D01-D03: vacancy and Pb-adatom defect calculations
#   G01-G03: clean Pb(111) calculations at selected fixed electron chemical potentials
#
# Common model:
#   3x3 Pb(111) surface cell
#   Optimized bulk Pb lattice constant
#   15 Å vacuum/fluid region on each side of the slab
#   Bottom two Pb layers fixed
#   PBE+D3 / CANDLE + 0.5 M NaF via go.py
#
# Notes:
#   target-mu values correspond approximately to -1.0, -1.4, and -1.8 V vs RHE
#   at pH 7 under the historical potential convention used for this Pb project.
#   target-mu is the authoritative raw electrochemical control variable.
#   Directory names retain target-mu as the primary raw-calculation identifier.
#   This cell refuses to overwrite existing calculation directories.

from pathlib import Path
import json
import re

import numpy as np
from ase.build import add_adsorbate, fcc111
from ase.constraints import FixAtoms
from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

BULK_PATH = ROOT / "bulk" / "Pb_fcc" / "CONTCAR"
CLEAN_PARENT_PATH = ROOT / "structures" / "Pb111" / "clean" / "POSCAR"
GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

VACUUM_A = 15.0

for path in [BULK_PATH, CLEAN_PARENT_PATH, GO_TEMPLATE, SUBMIT_TEMPLATE]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")

bulk = read(BULK_PATH, format="vasp")
lengths = bulk.cell.lengths()

if not np.allclose(lengths, lengths[0], atol=1e-5):
    raise ValueError(f"Expected cubic optimized Pb bulk cell, got lengths: {lengths}")

a = float(lengths[0])
d111 = a / np.sqrt(3.0)

clean_parent = read(CLEAN_PARENT_PATH, format="vasp")

if len(clean_parent) != 36:
    raise ValueError(f"Expected 36 Pb atoms in clean 3x3x4 parent, found {len(clean_parent)}")

if set(clean_parent.get_chemical_symbols()) != {"Pb"}:
    raise ValueError("Clean parent contains elements other than Pb")


def constrain_bottom_two(slab):
    tags = slab.get_tags()
    n_layers = int(tags.max())
    slab.set_constraint(FixAtoms(mask=tags >= n_layers - 1))
    return slab


def make_111(layers=4):
    if layers == 4:
        return clean_parent.copy()

    slab = fcc111(
        "Pb",
        size=(3, 3, layers),
        a=a,
        vacuum=VACUUM_A,
        orthogonal=False,
    )
    return constrain_bottom_two(slab)


def setup_job(
    job_id,
    relpath,
    atoms,
    job_name,
    purpose,
    family,
    kpts=(4, 4, 1, "gamma"),
    target_mu=None,
    approx_U_RHE_V=None,
    maxstep=None,
    parent="structures/Pb111/clean/POSCAR",
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    kpt_text = f"({kpts[0]}, {kpts[1]}, {kpts[2]}, {kpts[3]!r})"
    go_text, n = re.subn(r"kpts=\([^)]*\)", f"kpts={kpt_text}", go_text, count=1)
    if n != 1:
        raise RuntimeError("Could not uniquely replace kpts in go.py")

    if target_mu is not None:
        marker = "fluid-anion F- 0.5\n"
        if marker not in go_text:
            raise RuntimeError("Could not locate electrolyte block in go.py")
        go_text = go_text.replace(marker, marker + f"target-mu {target_mu:.6f}\n", 1)

    if maxstep is not None:
        go_text, n = re.subn(r"maxstep\s*=\s*[0-9.]+", f"maxstep={maxstep}", go_text, count=1)
        if n != 1:
            raise RuntimeError("Could not uniquely replace FIRE maxstep in go.py")

    submit_text, n = re.subn(
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        submit_text,
        count=1,
        flags=re.MULTILINE,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name")

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": family,
        "purpose": purpose,
        "parent": parent,
        "bulk_lattice_constant_A": a,
        "vacuum_each_side_A": VACUUM_A,
        "kpts": list(kpts),
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "ase_pbc": [bool(x) for x in atoms.pbc],
        "n_atoms": len(atoms),
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(f"{job_id:>3}  {job_dir}")


# ---------------------------------------------------------------------
# S01-S04: clean Pb(111) and numerical convergence
# ---------------------------------------------------------------------

setup_job(
    "S01",
    "surfaces/Pb111/clean",
    make_111(4),
    "Pb111_clean",
    "Primary clean Pb(111) parent: 4 layers, 4x4x1 k-point mesh.",
    family="clean_surface",
)

setup_job(
    "S02",
    "surfaces/Pb111/convergence/4L_k6",
    make_111(4),
    "Pb111_4L_k6",
    "Clean Pb(111), 4 layers, denser 6x6x1 k-point sampling.",
    family="surface_convergence",
    kpts=(6, 6, 1, "gamma"),
)

setup_job(
    "S03",
    "surfaces/Pb111/convergence/5L_k4",
    make_111(5),
    "Pb111_5L_k4",
    "Clean Pb(111), 5 layers, slab-thickness comparison.",
    family="surface_convergence",
    parent="bulk/Pb_fcc/CONTCAR",
)

setup_job(
    "S04",
    "surfaces/Pb111/convergence/6L_k4",
    make_111(6),
    "Pb111_6L_k4",
    "Clean Pb(111), 6 layers, slab-thickness comparison.",
    family="surface_convergence",
    parent="bulk/Pb_fcc/CONTCAR",
)


# ---------------------------------------------------------------------
# H01-H04: one-H adsorption-site screening
# ---------------------------------------------------------------------

h_sites = {
    "H01": ("ontop", 1.8),
    "H02": ("bridge", 1.0),
    "H03": ("fcc", 1.0),
    "H04": ("hcp", 1.0),
}

for job_id, (site, height) in h_sites.items():
    slab = make_111(4)
    add_adsorbate(slab, "H", height=height, position=site, offset=(1, 1))

    setup_job(
        job_id,
        f"surfaces/Pb111/H_adsorption/{site}",
        slab,
        f"Pb111_H_{site}",
        f"One H initially adsorbed at the Pb(111) {site} site.",
        family="H_adsorption",
        maxstep=0.10,
    )


# ---------------------------------------------------------------------
# D01-D03: simple corrosion/undercoordinated Pb models
# ---------------------------------------------------------------------

vacancy = make_111(4)
top_indices = np.where(vacancy.get_tags() == 1)[0]
vacancy_index = int(top_indices[len(top_indices) // 2])
del vacancy[vacancy_index]

setup_job(
    "D01",
    "surfaces/Pb111/defects/vacancy",
    vacancy,
    "Pb111_vac",
    "Pb(111) with one top-layer Pb vacancy; corrosion/extraction endpoint reference.",
    family="Pb_defect",
)

for job_id, site in [("D02", "fcc"), ("D03", "hcp")]:
    slab = make_111(4)
    add_adsorbate(slab, "Pb", height=d111, position=site, offset=(1, 1))

    setup_job(
        job_id,
        f"surfaces/Pb111/defects/adatom_{site}",
        slab,
        f"Pb111_ad_{site}",
        f"Pb adatom initially placed at the Pb(111) {site} hollow.",
        family="Pb_defect",
    )


# ---------------------------------------------------------------------
# G01-G03: clean Pb(111) at selected fixed electron chemical potentials
# ---------------------------------------------------------------------

fixed_mu_jobs = [
    ("G01", -0.1193, -1.0),
    ("G02", -0.1046, -1.4),
    ("G03", -0.0899, -1.8),
]

for job_id, mu, U_RHE in fixed_mu_jobs:
    mu_label = f"m{abs(mu):.4f}".replace(".", "p")

    setup_job(
        job_id,
        f"surfaces/Pb111/fixed_mu/{mu_label}",
        make_111(4),
        f"Pb111_{mu_label}",
        f"Clean Pb(111) relaxed at target-mu = {mu:.4f} Ha.",
        family="fixed_mu_clean",
        target_mu=mu,
        approx_U_RHE_V=U_RHE,
    )


print()
print(f"Bulk lattice constant used: {a:.8f} Å")
print(f"Vacuum / fluid-region spacing on each side: {VACUUM_A:.1f} Å")
print(f"Pb(111) interlayer spacing: {d111:.8f} Å")
print("Wave 1 Pb(111) batch setup complete.")

In [ ]:
# Purpose: Generate Wave 2 Pb(111) hydrogen structures.
#
# Inputs:
#   bulk/Pb_fcc/CONTCAR
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   SH01-SH03: candidate first-subsurface H structures
#   HH01-HH03: candidate two-H / PbH2-precursor structures
#
# Geometry:
#   3x3 Pb(111), 4 layers, 15 Å vacuum/fluid region on each side
#   bottom two Pb layers fixed
#
# Notes:
#   These calculations are intentionally generated from the common ideal Pb(111)
#   geometry derived from the optimized bulk lattice constant, rather than from
#   relaxed S01, so they can run in parallel with Wave 1.
#   Site labels describe INITIAL geometry only. Final relaxed structures must be
#   classified afterward.

from pathlib import Path
import json
import re

import numpy as np
from ase import Atom
from ase.build import add_adsorbate, fcc111
from ase.constraints import FixAtoms
from ase.io import read, write
from ase.geometry import find_mic

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

BULK_PATH = ROOT / "bulk" / "Pb_fcc" / "CONTCAR"
GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

VACUUM_A = 15.0

for path in [BULK_PATH, GO_TEMPLATE, SUBMIT_TEMPLATE]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")

bulk = read(BULK_PATH, format="vasp")
lengths = bulk.cell.lengths()

if not np.allclose(lengths, lengths[0], atol=1e-5):
    raise ValueError(f"Expected cubic optimized Pb bulk cell, got lengths: {lengths}")

a = float(lengths[0])


def make_111():
    slab = fcc111(
        "Pb",
        size=(3, 3, 4),
        a=a,
        vacuum=VACUUM_A,
        orthogonal=False,
    )
    tags = slab.get_tags()
    slab.set_constraint(FixAtoms(mask=tags >= 3))
    return slab


def setup_job(job_id, relpath, atoms, job_name, purpose, family, maxstep=0.10):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    go_text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={maxstep}",
        go_text,
        count=1,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep in go.py")

    submit_text, n = re.subn(
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        submit_text,
        count=1,
        flags=re.MULTILINE,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name")

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": family,
        "purpose": purpose,
        "parent": "bulk/Pb_fcc/CONTCAR",
        "bulk_lattice_constant_A": a,
        "vacuum_each_side_A": VACUUM_A,
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [bool(x) for x in atoms.pbc],
        "n_atoms": len(atoms),
        "initial_geometry_only": True,
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(f"{job_id:>4}  {job_dir}")


def adsorption_xy(site):
    probe = make_111()
    add_adsorbate(probe, "H", height=1.0, position=site, offset=(1, 1))
    return probe.positions[-1, :2].copy()


# First- and second-Pb-layer z coordinates
reference = make_111()
tags = reference.get_tags()

z_top = float(reference.positions[tags == 1, 2].mean())
z_second = float(reference.positions[tags == 2, 2].mean())
z_sub = 0.5 * (z_top + z_second)

print(f"Top-layer z:       {z_top:.6f} Å")
print(f"Second-layer z:    {z_second:.6f} Å")
print(f"Subsurface H z0:   {z_sub:.6f} Å")


# ---------------------------------------------------------------------
# SH01-SH03: first-subsurface H candidate positions
# ---------------------------------------------------------------------

for job_id, site in [
    ("SH01", "fcc"),
    ("SH02", "hcp"),
    ("SH03", "ontop"),
]:
    slab = make_111()
    xy = adsorption_xy(site)

    slab += Atom("H", position=[xy[0], xy[1], z_sub])

    setup_job(
        job_id,
        f"surfaces/Pb111/H_subsurface/{site}",
        slab,
        f"Pb111_Hsub_{site}",
        (
            "One H initially placed halfway between the first and second Pb layers, "
            f"laterally registered with the {site} surface site."
        ),
        family="H_subsurface",
        maxstep=0.08,
    )


# ---------------------------------------------------------------------
# HH01: ordinary two-H surface reference
# ---------------------------------------------------------------------

slab = make_111()

add_adsorbate(
    slab,
    "H",
    height=1.0,
    position="fcc",
    offset=(1, 1),
)

add_adsorbate(
    slab,
    "H",
    height=1.0,
    position="hcp",
    offset=(1, 1),
)

setup_job(
    "HH01",
    "surfaces/Pb111/H2_candidates/fcc_hcp",
    slab,
    "Pb111_2H_fcc_hcp",
    "Two surface H atoms initially occupying neighboring FCC and HCP hollow environments.",
    family="H2_candidate",
)


# ---------------------------------------------------------------------
# HH02: two bridge H atoms sharing the same central surface Pb
# ---------------------------------------------------------------------

slab = make_111()

tags = slab.get_tags()
top_indices = np.where(tags == 1)[0]

center_xy = 0.5 * (slab.cell[0, :2] + slab.cell[1, :2])
center_index = min(
    top_indices,
    key=lambda i: np.linalg.norm(slab.positions[i, :2] - center_xy),
)

center = slab.positions[center_index].copy()

neighbor_data = []

for i in top_indices:
    if i == center_index:
        continue

    vec = slab.positions[i] - center
    mic_vec, dist = find_mic(vec, slab.cell, pbc=slab.pbc)

    neighbor_data.append(
        (
            float(dist),
            int(i),
            mic_vec,
        )
    )

neighbor_data.sort(key=lambda x: x[0])

# Select two nearest neighbors that are not collinear periodic images.
n1 = neighbor_data[0]

n2 = next(
    item
    for item in neighbor_data[1:]
    if abs(np.dot(n1[2], item[2]) / (n1[0] * item[0])) < 0.9
)

for _, _, vec in [n1, n2]:
    bridge = center + 0.5 * vec
    bridge[2] += 1.0

    slab += Atom("H", position=bridge)

setup_job(
    "HH02",
    "surfaces/Pb111/H2_candidates/shared_bridges",
    slab,
    "Pb111_2H_bridges",
    "Two H atoms initially placed at two bridge sites sharing the same top-layer Pb atom.",
    family="H2_candidate",
)


# ---------------------------------------------------------------------
# HH03: direct PbH2-like local geometry
# ---------------------------------------------------------------------

slab = make_111()

tags = slab.get_tags()
top_indices = np.where(tags == 1)[0]

center_xy = 0.5 * (slab.cell[0, :2] + slab.cell[1, :2])
center_index = min(
    top_indices,
    key=lambda i: np.linalg.norm(slab.positions[i, :2] - center_xy),
)

pb = slab.positions[center_index].copy()

r_PbH = 1.90
theta = np.deg2rad(45.0)

dx = r_PbH * np.sin(theta)
dz = r_PbH * np.cos(theta)

slab += Atom("H", position=pb + [dx, 0.0, dz])
slab += Atom("H", position=pb + [-dx, 0.0, dz])

setup_job(
    "HH03",
    "surfaces/Pb111/H2_candidates/PbH2_like",
    slab,
    "Pb111_PbH2_like",
    (
        "Two H atoms initially arranged as a bent PbH2-like moiety on one top-layer "
        "Pb atom; Pb-H = 1.90 Å initial guess."
    ),
    family="H2_candidate",
    maxstep=0.06,
)

print()
print(f"Bulk lattice constant used: {a:.8f} Å")
print(f"Vacuum / fluid-region spacing on each side: {VACUUM_A:.1f} Å")
print("Wave 2 setup complete.")

In [ ]:
# Purpose: Generate Wave 3 fixed-potential Pb(111) H and 2H calculations.
#
# Inputs:
#   H01-H04 initial POSCAR structures from Wave 1
#   HH01-HH03 initial POSCAR structures from Wave 2
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   MH01-MH12: four one-H starting geometries at three target-mu values
#   M2H01-M2H09: three two-H starting geometries at three target-mu values
#
# Potentials:
#   target-mu = -0.1193 Ha  ~ -1.0 V vs RHE at pH 7
#   target-mu = -0.1046 Ha  ~ -1.4 V vs RHE at pH 7
#   target-mu = -0.0899 Ha  ~ -1.8 V vs RHE at pH 7
#
# Notes:
#   target-mu is the authoritative raw electrochemical control variable.
#   Approximate RHE potentials are historical metadata only.
#   These calculations use the GENERATED INITIAL H/2H structures rather than
#   neutral relaxed CONTCARs, so Wave 3 can run in parallel with Waves 1 and 2.
#   Final relaxed structures must be classified afterward because fixed-potential
#   relaxation can substantially change the initial structural motif.

from pathlib import Path
import json
import re

from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

MU_STATES = [
    (-0.1193, -1.0),
    (-0.1046, -1.4),
    (-0.0899, -1.8),
]

H_PARENTS = {
    "ontop": {
        "job_id": "H01",
        "path": ROOT / "surfaces/Pb111/H_adsorption/ontop/POSCAR",
    },
    "bridge": {
        "job_id": "H02",
        "path": ROOT / "surfaces/Pb111/H_adsorption/bridge/POSCAR",
    },
    "fcc": {
        "job_id": "H03",
        "path": ROOT / "surfaces/Pb111/H_adsorption/fcc/POSCAR",
    },
    "hcp": {
        "job_id": "H04",
        "path": ROOT / "surfaces/Pb111/H_adsorption/hcp/POSCAR",
    },
}

H2_PARENTS = {
    "fcc_hcp": {
        "job_id": "HH01",
        "path": ROOT / "surfaces/Pb111/H2_candidates/fcc_hcp/POSCAR",
    },
    "shared_bridges": {
        "job_id": "HH02",
        "path": ROOT / "surfaces/Pb111/H2_candidates/shared_bridges/POSCAR",
    },
    "PbH2_like": {
        "job_id": "HH03",
        "path": ROOT / "surfaces/Pb111/H2_candidates/PbH2_like/POSCAR",
    },
}

for path in [
    GO_TEMPLATE,
    SUBMIT_TEMPLATE,
    *(entry["path"] for entry in H_PARENTS.values()),
    *(entry["path"] for entry in H2_PARENTS.values()),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")


def mu_label(mu):
    return f"m{abs(mu):.4f}".replace(".", "p")


def setup_job(
    job_id,
    family,
    source_job_id,
    parent_path,
    relpath,
    job_name,
    purpose,
    mu,
    U_RHE,
    maxstep,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    atoms = read(parent_path, format="vasp")
    atoms.set_pbc((True, True, False))

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    marker = "fluid-anion F- 0.5\n"
    if marker not in go_text:
        raise RuntimeError("Could not locate electrolyte block in go.py")

    if "target-mu" in go_text:
        raise RuntimeError("Canonical go.py unexpectedly already contains target-mu")

    go_text = go_text.replace(
        marker,
        marker + f"target-mu {mu:.6f}\n",
        1,
    )

    go_text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={maxstep}",
        go_text,
        count=1,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep")

    submit_text, n = re.subn(
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        submit_text,
        count=1,
        flags=re.MULTILINE,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name")

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": family,
        "purpose": purpose,
        "source_job_id": source_job_id,
        "source_path": str(parent_path.relative_to(ROOT)),
        "target_mu_Ha": mu,
        "approx_U_RHE_V_at_pH7": U_RHE,
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [True, True, False],
        "n_atoms": len(atoms),
        "initial_geometry_only": True,
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(
        f"{job_id:>5}  source={source_job_id:<4}  "
        f"mu={mu: .5f}  {job_dir}"
    )


# ---------------------------------------------------------------------
# MH01-MH12: one-H fixed-potential calculations
# ---------------------------------------------------------------------

job_counter = 1

for site, parent in H_PARENTS.items():
    for mu, U_RHE in MU_STATES:
        mlab = mu_label(mu)
        job_id = f"MH{job_counter:02d}"

        setup_job(
            job_id=job_id,
            family="fixed_mu_1H",
            source_job_id=parent["job_id"],
            parent_path=parent["path"],
            relpath=f"surfaces/Pb111/fixed_mu/H_adsorption/{site}/{mlab}",
            job_name=f"Pb111_H_{site}_{mlab}",
            purpose=f"One-H Pb(111) {site} geometry relaxed at target-mu = {mu:.4f} Ha.",
            mu=mu,
            U_RHE=U_RHE,
            maxstep=0.08,
        )

        job_counter += 1


# ---------------------------------------------------------------------
# M2H01-M2H09: two-H fixed-potential calculations
# ---------------------------------------------------------------------

job_counter = 1

for geometry, parent in H2_PARENTS.items():
    for mu, U_RHE in MU_STATES:
        mlab = mu_label(mu)
        job_id = f"M2H{job_counter:02d}"

        setup_job(
            job_id=job_id,
            family="fixed_mu_2H",
            source_job_id=parent["job_id"],
            parent_path=parent["path"],
            relpath=f"surfaces/Pb111/fixed_mu/H2_candidates/{geometry}/{mlab}",
            job_name=f"Pb111_2H_{geometry[:8]}_{mlab}",
            purpose=f"Two-H candidate '{geometry}' relaxed at target-mu = {mu:.4f} Ha.",
            mu=mu,
            U_RHE=U_RHE,
            maxstep=0.06,
        )

        job_counter += 1


print()
print("Wave 3 setup complete: 21 fixed-potential H/2H calculations.")

In [ ]:
# Purpose: Generate Wave 4 PbH2-extraction endpoint searches, the fixed-potential
# deep-subsurface-H test, and the first four NEB calculations.
#
# Inputs:
#   HH02 CONTCAR: neutral same-Pb two-H state
#   M2H07 CONTCAR: strongly lifted same-Pb two-H state at target-mu = -0.1193 Ha
#   SH03 CONTCAR: deep subsurface-H minimum
#   Relaxed endpoint calculations for N01-N04
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#   Pb_LAR_source/control/neb.py
#   Pb_LAR_source/control/submit_neb.sh
#   Pb_LAR_source/scripts/setup_neb_images.py
#
# Outputs:
#   X01-X03: neutral PbH2 extraction endpoint searches
#   X04-X06: fixed-potential PbH2 extraction endpoint searches at -0.1193 Ha
#   X07: fixed-potential continuation of the deep subsurface-H minimum
#   N01-N04: initial non-climbing NEB bands
#
# NEB setup:
#   8 intermediate images
#   ASE IDPP interpolation with MIC=True
#   first pass deliberately non-climbing
#   FIRE maxstep = 0.05 Å
#   fmax = 0.08 eV/Å
#   maximum 150 optimization steps
#
# Notes:
#   target-mu is the authoritative electrochemical control variable.
#   The approximate -1.0 V vs RHE assignment is historical metadata only.
#   Fixed-potential NEB endpoint energies are read from the JDFTx G entry;
#   neutral endpoint energies are read from F.
#   This cell refuses to overwrite existing calculation directories.

from pathlib import Path
import json
import re
import shutil
import subprocess
import sys

import numpy as np
from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_GO_TEMPLATE = CONTROL / "submit_go.sh"
NEB_TEMPLATE = CONTROL / "neb.py"
SUBMIT_NEB_TEMPLATE = CONTROL / "submit_neb.sh"
NEB_IMAGE_SCRIPT = SOURCE_ROOT / "scripts" / "setup_neb_images.py"

TARGET_MU = -0.1193
APPROX_U_RHE_V = -1.0
NIMAGES = 8

for path in [
    GO_TEMPLATE,
    SUBMIT_GO_TEMPLATE,
    NEB_TEMPLATE,
    SUBMIT_NEB_TEMPLATE,
    NEB_IMAGE_SCRIPT,
]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required source file: {path}")


def replace_once(text, pattern, replacement, description, flags=0):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=flags)

    if n != 1:
        raise RuntimeError(f"Could not uniquely replace {description}")

    return new_text


def surface_z(atoms):
    symbols = np.array(atoms.get_chemical_symbols())
    pb_indices = np.where(symbols == "Pb")[0]

    if len(pb_indices) < 9:
        raise ValueError(f"Expected at least 9 Pb atoms, found {len(pb_indices)}")

    pb_z = atoms.positions[pb_indices, 2]
    return float(np.median(np.sort(pb_z)[-9:]))


def shared_pb_h2_indices(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(h_indices) != 2:
        raise ValueError(f"Expected exactly 2 H atoms, found {len(h_indices)}")

    nearest_pb = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(h_idx), int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ])

        nearest_local = int(np.argmin(distances))
        nearest_pb.append(int(pb_indices[nearest_local]))

        if distances[nearest_local] > 2.6:
            raise ValueError(
                f"H atom {h_idx} is not clearly Pb-bound: nearest Pb-H = "
                f"{distances[nearest_local]:.3f} Å"
            )

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two H atoms do not share one Pb: nearest Pb = {nearest_pb}"
        )

    return nearest_pb[0], [int(i) for i in h_indices]


def make_extraction_structure(parent_path, target_pb_lift_A):
    atoms = read(parent_path, format="vasp")
    atoms.set_pbc((True, True, False))

    pb_idx, h_indices = shared_pb_h2_indices(atoms)

    z_surface = surface_z(atoms)
    current_lift = float(atoms.positions[pb_idx, 2] - z_surface)
    dz = target_pb_lift_A - current_lift

    move_indices = [pb_idx, *h_indices]
    atoms.positions[move_indices, 2] += dz

    top_gap = float(atoms.cell[2, 2] - np.max(atoms.positions[:, 2]))

    if top_gap < 4.0:
        raise ValueError(
            f"Only {top_gap:.2f} Å remains above the highest atom. "
            "Increase the cell height before using this structure."
        )

    return atoms, {
        "shared_Pb_index": pb_idx,
        "H_indices": h_indices,
        "initial_Pb_lift_A": current_lift,
        "target_Pb_lift_A": target_pb_lift_A,
        "rigid_translation_A": dz,
        "top_gap_A": top_gap,
    }


def setup_relaxation(
    job_id,
    family,
    atoms,
    relpath,
    job_name,
    purpose,
    source_job_id,
    source_path,
    target_mu=None,
    approx_U_RHE_V=None,
    extra_metadata=None,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    atoms = atoms.copy()
    atoms.set_pbc((True, True, False))

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_GO_TEMPLATE.read_text()

    go_text = replace_once(
        go_text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "FIRE maxstep",
    )

    if target_mu is not None:
        if "target-mu" in go_text:
            raise RuntimeError("Canonical go.py unexpectedly already contains target-mu")

        marker = "fluid-anion F- 0.5\n"

        if marker not in go_text:
            raise RuntimeError("Could not locate electrolyte block in go.py")

        go_text = go_text.replace(
            marker,
            marker + f"target-mu {target_mu:.6f}\n",
            1,
        )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "Slurm job name",
        flags=re.MULTILINE,
    )

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": family,
        "purpose": purpose,
        "source_job_id": source_job_id,
        "source_path": str(source_path),
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [True, True, False],
        "n_atoms": len(atoms),
    }

    if extra_metadata:
        metadata.update(extra_metadata)

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(
        f"{job_id:>5}  source={source_job_id:<5}  "
        f"mu={str(target_mu):>8}  {job_dir}"
    )


def latest_endpoint_file(source_dir, suffix):
    candidates = sorted(
        source_dir.glob(f"*{suffix}"),
        key=lambda path: path.stat().st_mtime,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No *{suffix} file found in endpoint directory: {source_dir}"
        )

    return candidates[-1]


def endpoint_files(source_dir):
    contcar = source_dir / "CONTCAR"

    if not contcar.is_file():
        raise FileNotFoundError(f"Missing endpoint CONTCAR: {contcar}")

    return {
        "CONTCAR": contcar,
        "Ecomponents": latest_endpoint_file(source_dir, "Ecomponents"),
        "force": latest_endpoint_file(source_dir, "force"),
    }


def copy_endpoint(source_files, target_dir):
    shutil.copy2(source_files["CONTCAR"], target_dir / "CONTCAR")
    shutil.copy2(
        source_files["Ecomponents"],
        target_dir / source_files["Ecomponents"].name,
    )
    shutil.copy2(
        source_files["force"],
        target_dir / source_files["force"].name,
    )


def setup_neb(
    job_id,
    family,
    initial_job_id,
    final_job_id,
    initial_dir,
    final_dir,
    relpath,
    job_name,
    purpose,
    target_mu=None,
    approx_U_RHE_V=None,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    initial_files = endpoint_files(initial_dir)
    final_files = endpoint_files(final_dir)

    final_index = NIMAGES + 1

    job_dir.mkdir(parents=True)

    initial_image_dir = job_dir / "00"
    final_image_dir = job_dir / f"{final_index:02d}"

    initial_image_dir.mkdir()
    final_image_dir.mkdir()

    copy_endpoint(initial_files, initial_image_dir)
    copy_endpoint(final_files, final_image_dir)

    # Canonical project utility: validates endpoints and creates MIC-aware IDPP images.
    subprocess.run(
        [
            sys.executable,
            str(NEB_IMAGE_SCRIPT),
            "--dir",
            str(job_dir),
            "--nimages",
            str(NIMAGES),
        ],
        check=True,
    )

    neb_text = NEB_TEMPLATE.read_text()
    submit_text = SUBMIT_NEB_TEMPLATE.read_text()

    energy_label = "G" if target_mu is not None else "F"

    old_energy_function = re.compile(
        r"def _read_endpoint_energy\(s\):.*?"
        r"(?=\ndef _read_endpoint_forces)",
        flags=re.DOTALL,
    )

    new_energy_function = f'''def _read_endpoint_energy(s):
    f = next(n for n in os.listdir(s) if n.endswith('Ecomponents'))
    label = "{energy_label}"
    with open(os.path.join(s, f)) as fh:
        lines = [ln.strip() for ln in fh if ln.strip()]
    for line in reversed(lines):
        fields = line.split()
        if len(fields) >= 3 and fields[0] == label and fields[1] == '=':
            return float(fields[2]) * Hartree
    raise RuntimeError(f"Could not find {{label}} in {{os.path.join(s, f)}}")
'''

    neb_text, n = old_energy_function.subn(
        new_energy_function,
        neb_text,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not replace NEB endpoint-energy parser")

    neb_text = replace_once(
        neb_text,
        r"^NIMAGES\s*=\s*\d+\s*$",
        f"NIMAGES = {NIMAGES}",
        "NEB image count",
        flags=re.MULTILINE,
    )

    if target_mu is not None:
        if "target-mu" in neb_text:
            raise RuntimeError("Canonical neb.py unexpectedly already contains target-mu")

        marker = "fluid-anion F- 0.5\n"

        if marker not in neb_text:
            raise RuntimeError("Could not locate electrolyte block in neb.py")

        neb_text = neb_text.replace(
            marker,
            marker + f"target-mu {target_mu:.6f}\n",
            1,
        )

    neb_text = replace_once(
        neb_text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "NEB FIRE maxstep",
    )

    neb_text = replace_once(
        neb_text,
        r"opt\.run\(fmax\s*=\s*[0-9.]+,\s*steps\s*=\s*\d+\)",
        "opt.run(fmax=0.08, steps=150)",
        "NEB optimizer settings",
    )

    if "climb=False" not in neb_text:
        raise RuntimeError("Expected non-climbing NEB template with climb=False")

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH -q .*?$",
        "#SBATCH -q regular",
        "NEB QoS",
        flags=re.MULTILINE,
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --time .*?$",
        "#SBATCH --time 08:00:00",
        "NEB walltime",
        flags=re.MULTILINE,
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "NEB job name",
        flags=re.MULTILINE,
    )

    (job_dir / "neb.py").write_text(neb_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": family,
        "purpose": purpose,
        "initial_job_id": initial_job_id,
        "final_job_id": final_job_id,
        "initial_endpoint": str(initial_dir.relative_to(ROOT)),
        "final_endpoint": str(final_dir.relative_to(ROOT)),
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "endpoint_energy_label": energy_label,
        "n_intermediate_images": NIMAGES,
        "interpolation": "ASE IDPP",
        "mic": True,
        "climb": False,
        "fmax_eV_A": 0.08,
        "maxstep_A": 0.05,
        "max_steps": 150,
        "stage": "initial_band_relaxation",
        "setup_script": str(NEB_IMAGE_SCRIPT.relative_to(SOURCE_ROOT)),
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(
        f"{job_id:>5}  {initial_job_id} -> {final_job_id}  "
        f"mu={str(target_mu):>8}  {job_dir}"
    )


# ---------------------------------------------------------------------
# X01-X06: PbH2 extraction endpoint searches
# ---------------------------------------------------------------------

neutral_parent = (
    ROOT
    / "surfaces/Pb111/H2_candidates/shared_bridges/CONTCAR"
)

fixed_mu_parent = (
    ROOT
    / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193/CONTCAR"
)

for path in [neutral_parent, fixed_mu_parent]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required relaxed parent: {path}")


# X01-X03: neutral extraction from HH02
for i, target_lift in enumerate((2.5, 4.0, 6.0), start=1):
    atoms, info = make_extraction_structure(
        neutral_parent,
        target_pb_lift_A=target_lift,
    )

    label = str(target_lift).replace(".", "p")

    setup_relaxation(
        job_id=f"X{i:02d}",
        family="PbH2_extraction",
        atoms=atoms,
        relpath=f"surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_{label}",
        job_name=f"PbH2_N_z{label}",
        purpose=(
            "Neutral PbH2-extraction endpoint search with the Pb(H)2 moiety "
            f"initially translated so its Pb atom is {target_lift:.1f} Å above "
            "the median Pb(111) surface plane."
        ),
        source_job_id="HH02",
        source_path=neutral_parent.relative_to(ROOT),
        target_mu=None,
        extra_metadata=info,
    )


# X04-X06: fixed-potential extraction from M2H07
for i, target_lift in enumerate((2.5, 4.0, 6.0), start=4):
    atoms, info = make_extraction_structure(
        fixed_mu_parent,
        target_pb_lift_A=target_lift,
    )

    label = str(target_lift).replace(".", "p")

    setup_relaxation(
        job_id=f"X{i:02d}",
        family="PbH2_extraction",
        atoms=atoms,
        relpath=(
            "surfaces/Pb111/PbH2_extraction/fixed_mu/"
            f"m0p1193/Pb_lift_{label}"
        ),
        job_name=f"PbH2_mu1193_z{label}",
        purpose=(
            "Fixed-potential PbH2-extraction endpoint search at target-mu "
            f"{TARGET_MU:.4f} Ha with the Pb(H)2 moiety initially translated "
            f"so its Pb atom is {target_lift:.1f} Å above the median Pb(111) "
            "surface plane."
        ),
        source_job_id="M2H07",
        source_path=fixed_mu_parent.relative_to(ROOT),
        target_mu=TARGET_MU,
        approx_U_RHE_V=APPROX_U_RHE_V,
        extra_metadata=info,
    )


# ---------------------------------------------------------------------
# X07: fixed-potential continuation of the deep subsurface-H minimum
# ---------------------------------------------------------------------

subsurface_parent = (
    ROOT
    / "surfaces/Pb111/H_subsurface/ontop/CONTCAR"
)

if not subsurface_parent.is_file():
    raise FileNotFoundError(f"Missing required relaxed parent: {subsurface_parent}")

subsurface_atoms = read(subsurface_parent, format="vasp")
subsurface_atoms.set_pbc((True, True, False))

setup_relaxation(
    job_id="X07",
    family="subsurface_H",
    atoms=subsurface_atoms,
    relpath="surfaces/Pb111/fixed_mu/H_subsurface/deep/m0p1193",
    job_name="Pb111_Hsub_mu1193",
    purpose=(
        "Test whether the deep subsurface-H / Pb-lifted structure remains a "
        "local minimum at target-mu = -0.1193 Ha."
    ),
    source_job_id="SH03",
    source_path=subsurface_parent.relative_to(ROOT),
    target_mu=TARGET_MU,
    approx_U_RHE_V=APPROX_U_RHE_V,
)


# ---------------------------------------------------------------------
# N01-N04: first NEB set
# ---------------------------------------------------------------------

setup_neb(
    job_id="N01",
    family="fixed_mu_neb",
    initial_job_id="MH07",
    final_job_id="MH01",
    initial_dir=ROOT / "surfaces/Pb111/fixed_mu/H_adsorption/fcc/m0p1193",
    final_dir=ROOT / "surfaces/Pb111/fixed_mu/H_adsorption/ontop/m0p1193",
    relpath="kinetics/Pb111/fixed_mu/m0p1193/surface_H_to_lifted_PbH",
    job_name="NEB_H_lift_mu1193",
    purpose=(
        "Grand-canonical NEB from ordinary surface-bound H to the lifted Pb-H "
        "state at target-mu = -0.1193 Ha."
    ),
    target_mu=TARGET_MU,
    approx_U_RHE_V=APPROX_U_RHE_V,
)

setup_neb(
    job_id="N02",
    family="fixed_mu_neb",
    initial_job_id="M2H01",
    final_job_id="M2H04",
    initial_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/fcc_hcp/m0p1193",
    final_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1193",
    relpath="kinetics/Pb111/fixed_mu/m0p1193/separated_2H_to_same_Pb",
    job_name="NEB_2H_local_mu1193",
    purpose=(
        "Grand-canonical NEB from two H atoms associated with different Pb atoms "
        "to the same-Pb Pb(H)2 surface state at target-mu = -0.1193 Ha."
    ),
    target_mu=TARGET_MU,
    approx_U_RHE_V=APPROX_U_RHE_V,
)

setup_neb(
    job_id="N03",
    family="fixed_mu_neb",
    initial_job_id="M2H04",
    final_job_id="M2H07",
    initial_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1193",
    final_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193",
    relpath="kinetics/Pb111/fixed_mu/m0p1193/PbH2_surface_to_strongly_lifted",
    job_name="NEB_PbH2_lift_mu1193",
    purpose=(
        "Grand-canonical NEB from the mildly lifted same-Pb Pb(H)2 surface state "
        "to the strongly lifted incipient-extraction state at target-mu = -0.1193 Ha."
    ),
    target_mu=TARGET_MU,
    approx_U_RHE_V=APPROX_U_RHE_V,
)

setup_neb(
    job_id="N04",
    family="neutral_neb",
    initial_job_id="H03",
    final_job_id="SH03",
    initial_dir=ROOT / "surfaces/Pb111/H_adsorption/fcc",
    final_dir=ROOT / "surfaces/Pb111/H_subsurface/ontop",
    relpath="kinetics/Pb111/neutral/surface_H_to_deep_subsurface_H",
    job_name="NEB_H_sub_neutral",
    purpose=(
        "Neutral NEB from the lowest ordinary surface-H state to the deep "
        "subsurface-H / Pb-lifted minimum."
    ),
    target_mu=None,
)

print()
print("Wave 4 setup complete: 7 relaxations + 4 IDPP-interpolated NEBs.")

In [ ]:
# Purpose: Generate Wave 5 fixed-potential continuation of the strongly lifted
# surface Pb(H)2 precursor and detached PbH2 + vacancy states.
#
# Inputs:
#   M2H07 CONTCAR: strongly lifted surface Pb(H)2 at target-mu = -0.1193 Ha
#   X06 CONTCAR: detached PbH2 + Pb vacancy at target-mu = -0.1193 Ha
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   X08-X11: strongly lifted surface Pb(H)2 potential continuation
#   X12-X15: detached PbH2 + vacancy potential continuation
#   X16: farther-separation check for detached PbH2 at target-mu = -0.1193 Ha
#   Pb_LAR_source/campaigns/wave5_PbH2_potential_sweep.json
#
# Notes:
#   target-mu is the authoritative electrochemical control variable.
#   Approximate RHE potentials are historical metadata only.
#   Energetic comparison between precursor and detached states is meaningful only
#   when both relaxed structures remain physical states. Proton-well artifacts
#   must be classified and excluded before thermodynamic interpretation.
#   This cell refuses to overwrite existing calculation directories.

from pathlib import Path
import json
import re

import numpy as np
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

MANIFEST_DIR = SOURCE_ROOT / "campaigns"
MANIFEST_PATH = MANIFEST_DIR / "wave5_PbH2_potential_sweep.json"

PRECURSOR_PARENT = (
    ROOT
    / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193/CONTCAR"
)

DETACHED_PARENT = (
    ROOT
    / "surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_6p0/CONTCAR"
)

MU_POINTS = [
    ("X08", -0.11195, -1.2),
    ("X09", -0.10460, -1.4),
    ("X10", -0.09725, -1.6),
    ("X11", -0.08990, -1.8),
]

DETACHED_MU_POINTS = [
    ("X12", -0.11195, -1.2),
    ("X13", -0.10460, -1.4),
    ("X14", -0.09725, -1.6),
    ("X15", -0.08990, -1.8),
]


for path in [
    GO_TEMPLATE,
    SUBMIT_TEMPLATE,
    PRECURSOR_PARENT,
    DETACHED_PARENT,
]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")


def replace_once(text, pattern, replacement, description, flags=0):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=flags)

    if n != 1:
        raise RuntimeError(f"Could not uniquely replace {description}")

    return new_text


def mu_label(mu):
    value = f"{abs(mu):.5f}".rstrip("0").rstrip(".").replace(".", "p")
    return f"m{value}" if mu < 0 else f"p{value}"


def surface_z(atoms):
    symbols = np.array(atoms.get_chemical_symbols())
    pb_indices = np.where(symbols == "Pb")[0]

    if len(pb_indices) < 9:
        raise ValueError(f"Expected at least 9 Pb atoms, found {len(pb_indices)}")

    pb_z = atoms.positions[pb_indices, 2]
    return float(np.median(np.sort(pb_z)[-9:]))


def shared_pbh2_geometry(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(h_indices) != 2:
        raise ValueError(f"Expected exactly 2 H atoms, found {len(h_indices)}")

    nearest_pb = []
    nearest_distances = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(h_idx), int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))

        nearest_pb.append(int(pb_indices[local]))
        nearest_distances.append(float(distances[local]))

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"Two H atoms do not share one Pb: nearest Pb indices = {nearest_pb}"
        )

    if max(nearest_distances) > 2.4:
        raise ValueError(
            f"Expected intact Pb(H)2/PbH2 geometry; Pb-H distances = {nearest_distances}"
        )

    pb_idx = nearest_pb[0]
    z_surface = surface_z(atoms)
    pb_lift = float(atoms.positions[pb_idx, 2] - z_surface)

    return {
        "Pb_index": pb_idx,
        "H_indices": [int(i) for i in h_indices],
        "PbH_distances_A": nearest_distances,
        "Pb_lift_A": pb_lift,
    }


def setup_relaxation(
    job_id,
    atoms,
    relpath,
    job_name,
    purpose,
    source_job_id,
    source_path,
    target_mu,
    approx_U_RHE_V,
    state,
    extra_metadata=None,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    atoms = atoms.copy()
    atoms.set_pbc((True, True, False))

    job_dir.mkdir(parents=True)

    write(
        job_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    if re.search(r"(?m)^\s*target-mu\s+", go_text):
        raise RuntimeError("Canonical go.py unexpectedly already contains target-mu")

    marker = "fluid-anion F- 0.5\n"

    if marker not in go_text:
        raise RuntimeError("Could not locate electrolyte block in canonical go.py")

    go_text = go_text.replace(
        marker,
        marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    go_text = replace_once(
        go_text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "FIRE maxstep",
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "Slurm job name",
        flags=re.MULTILINE,
    )

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": "PbH2_potential_sweep",
        "state": state,
        "purpose": purpose,
        "source_job_id": source_job_id,
        "source_path": str(source_path.relative_to(ROOT)),
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [True, True, False],
        "n_atoms": len(atoms),
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": 0.05,
    }

    if extra_metadata:
        metadata.update(extra_metadata)

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(
        f"{job_id:>4}  source={source_job_id:<5}  "
        f"mu={target_mu: .5f}  {state:<38}  {job_dir}"
    )

    return {
        "job_id": job_id,
        "directory": str(job_dir.relative_to(ROOT)),
        "state": state,
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "source_job_id": source_job_id,
        "source_path": str(source_path.relative_to(ROOT)),
        "purpose": purpose,
    }


# ---------------------------------------------------------------------
# Load and validate Wave 5 parent states
# ---------------------------------------------------------------------

precursor = read(PRECURSOR_PARENT, format="vasp")
detached = read(DETACHED_PARENT, format="vasp")

precursor.set_pbc((True, True, False))
detached.set_pbc((True, True, False))

precursor_geometry = shared_pbh2_geometry(precursor)
detached_geometry = shared_pbh2_geometry(detached)

if not 0.5 < precursor_geometry["Pb_lift_A"] < 3.0:
    raise ValueError(
        "M2H07 does not resemble the expected strongly lifted surface Pb(H)2 state: "
        f"Pb lift = {precursor_geometry['Pb_lift_A']:.3f} Å"
    )

if detached_geometry["Pb_lift_A"] < 4.0:
    raise ValueError(
        "X06 does not resemble detached PbH2 + vacancy: "
        f"Pb lift = {detached_geometry['Pb_lift_A']:.3f} Å"
    )

print("Validated Wave 5 parents:")
print(
    f"  M2H07 precursor: Pb lift = {precursor_geometry['Pb_lift_A']:.3f} Å, "
    f"Pb-H = {precursor_geometry['PbH_distances_A']}"
)
print(
    f"  X06 detached:    Pb lift = {detached_geometry['Pb_lift_A']:.3f} Å, "
    f"Pb-H = {detached_geometry['PbH_distances_A']}"
)

jobs = []


# ---------------------------------------------------------------------
# X08-X11: strongly lifted surface Pb(H)2 potential continuation
# ---------------------------------------------------------------------

for job_id, mu, approx_U in MU_POINTS:
    label = mu_label(mu)

    jobs.append(
        setup_relaxation(
            job_id=job_id,
            atoms=precursor,
            relpath=(
                "surfaces/Pb111/PbH2_extraction/potential_sweep/"
                f"precursor/{label}"
            ),
            job_name=f"PbH2_pre_{label}",
            purpose=(
                "Potential-dependent relaxation of the strongly lifted "
                "surface Pb(H)2 precursor."
            ),
            source_job_id="M2H07",
            source_path=PRECURSOR_PARENT,
            target_mu=mu,
            approx_U_RHE_V=approx_U,
            state="strongly_lifted_surface_PbH2",
        )
    )


# ---------------------------------------------------------------------
# X12-X15: detached PbH2 + Pb vacancy potential continuation
# ---------------------------------------------------------------------

for job_id, mu, approx_U in DETACHED_MU_POINTS:
    label = mu_label(mu)

    jobs.append(
        setup_relaxation(
            job_id=job_id,
            atoms=detached,
            relpath=(
                "surfaces/Pb111/PbH2_extraction/potential_sweep/"
                f"detached/{label}"
            ),
            job_name=f"PbH2_det_{label}",
            purpose=(
                "Potential-dependent relaxation of detached PbH2 above "
                "a Pb(111) vacancy."
            ),
            source_job_id="X06",
            source_path=DETACHED_PARENT,
            target_mu=mu,
            approx_U_RHE_V=approx_U,
            state="detached_PbH2_plus_vacancy",
        )
    )


# ---------------------------------------------------------------------
# X16: separation-convergence check at target-mu = -0.1193 Ha
# ---------------------------------------------------------------------

farther = detached.copy()

pb_idx = detached_geometry["Pb_index"]
h_indices = detached_geometry["H_indices"]
initial_pb_h = detached_geometry["PbH_distances_A"]

translation_A = 1.5
move_indices = [pb_idx, *h_indices]

farther.positions[move_indices, 2] += translation_A

top_gap_A = float(farther.cell[2, 2] - np.max(farther.positions[:, 2]))

if top_gap_A < 4.0:
    raise ValueError(
        f"Only {top_gap_A:.2f} Å remains above the translated PbH2 unit; "
        "do not create X16 without increasing the cell height."
    )

jobs.append(
    setup_relaxation(
        job_id="X16",
        atoms=farther,
        relpath=(
            "surfaces/Pb111/PbH2_extraction/separation/"
            "fixed_mu/m0p1193/plus_1p5"
        ),
        job_name="PbH2_sep_mu1193",
        purpose=(
            "Separation-convergence check for detached PbH2 at "
            "target-mu = -0.1193 Ha; the intact PbH2 unit from X06 is "
            "translated 1.5 Å farther from the Pb(111) surface."
        ),
        source_job_id="X06",
        source_path=DETACHED_PARENT,
        target_mu=-0.11930,
        approx_U_RHE_V=-1.0,
        state="detached_PbH2_farther_from_surface",
        extra_metadata={
            "Pb_index_translated": pb_idx,
            "H_indices_translated": h_indices,
            "initial_PbH_distances_A": initial_pb_h,
            "rigid_translation_z_A": translation_A,
            "remaining_top_gap_A": top_gap_A,
        },
    )
)


# ---------------------------------------------------------------------
# Git-tracked Wave 5 campaign manifest
# ---------------------------------------------------------------------

manifest = {
    "wave": 5,
    "name": "PbH2 potential sweep and separation convergence",
    "manifest_role": "campaign_definition",
    "scientific_question": (
        "Test how cathodic target-mu affects the stability of the strongly lifted "
        "surface Pb(H)2 precursor and detached PbH2 + vacancy states. Where both "
        "relaxed states remain physical, compare their grand free energies at the "
        "same target-mu."
    ),
    "reference_states": [
        {
            "job_id": "M2H07",
            "state": "strongly_lifted_surface_PbH2",
            "target_mu_Ha": -0.1193,
        },
        {
            "job_id": "X06",
            "state": "detached_PbH2_plus_vacancy",
            "target_mu_Ha": -0.1193,
        },
    ],
    "primary_variable": "target_mu_Ha",
    "potential_mapping_note": (
        "approx_U_RHE_V_at_pH7 uses the historical project calibration and is "
        "not the authoritative calculation variable."
    ),
    "interpretation_note": (
        "Final relaxed structures must be classified before energetic comparison. "
        "Proton-well or otherwise unphysical states must not be included in the "
        "physical thermodynamic envelope."
    ),
    "jobs": jobs,
}

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():
    existing_manifest = json.loads(MANIFEST_PATH.read_text())

    if existing_manifest != manifest:
        raise RuntimeError(
            f"Existing Git-tracked manifest differs from this Wave 5 definition: "
            f"{MANIFEST_PATH}\nRefusing to overwrite it."
        )

    print(f"Manifest already exists and matches: {MANIFEST_PATH}")
else:
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"Wrote manifest: {MANIFEST_PATH}")

print()
print(f"Created {len(jobs)} Wave 5 jobs.")

In [ ]:
# Purpose: Generate Wave 6 surface-2H reference calculations and PbH2 extraction NEBs.
#
# Inputs:
#   M2H01: separated two-H surface state at target-mu = -0.1193 Ha
#   M2H04: same-Pb two-H surface state at target-mu = -0.1193 Ha
#   M2H05: physical separated two-H state at target-mu = -0.1046 Ha
#   HH02: neutral same-Pb two-H state
#   X01: neutral partially extracted Pb(H)2 minimum
#   X03: neutral detached PbH2 + vacancy
#   M2H07: strongly lifted Pb(H)2 state at target-mu = -0.1193 Ha
#   X16: asymptotically detached PbH2 + vacancy at target-mu = -0.1193 Ha
#   Pb_LAR_source/control/*
#   Pb_LAR_source/scripts/setup_neb_images.py
#
# Outputs:
#   X17-X20: physical surface-two-H reference searches at intermediate target-mu values
#   N05: HH02 -> X01 neutral Pb(H)2 partial-extraction NEB
#   N06: X01 -> X03 neutral PbH2 detachment NEB
#   N07: M2H07 -> X16 fixed-mu PbH2 detachment NEB
#   Pb_LAR_source/campaigns/wave6_surface2H_and_extraction_nebs.json
#
# NEB setup:
#   8 intermediate images
#   ASE IDPP interpolation with MIC=True
#   improved-tangent NEB
#   first pass deliberately non-climbing
#   FIRE maxstep = 0.05 Å
#   fmax = 0.08 eV/Å
#   maximum 150 FIRE steps
#
# Notes:
#   target-mu is the authoritative electrochemical control variable.
#   Approximate RHE potentials are historical metadata only.
#   State labels for X17-X20 describe the intended STARTING basin. Final relaxed
#   structures must be classified independently before energetic interpretation.
#   Fixed-mu NEB endpoint energies use G; neutral NEB endpoint energies use F.
#   This cell refuses to overwrite existing calculation directories.

from pathlib import Path
import json
import re
import shutil
import subprocess
import sys

import numpy as np
from ase.io import read


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_GO_TEMPLATE = CONTROL / "submit_go.sh"
NEB_TEMPLATE = CONTROL / "neb.py"
SUBMIT_NEB_TEMPLATE = CONTROL / "submit_neb.sh"

IDPP_SCRIPT = SOURCE_ROOT / "scripts" / "setup_neb_images.py"

MANIFEST_DIR = SOURCE_ROOT / "campaigns"
MANIFEST_PATH = MANIFEST_DIR / "wave6_surface2H_and_extraction_nebs.json"

NIMAGES = 8


# ---------------------------------------------------------------------
# Existing relaxed endpoint / parent calculations
# ---------------------------------------------------------------------

PATHS = {
    "M2H01": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/fcc_hcp/m0p1193",
    "M2H04": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1193",
    "M2H05": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1046",
    "HH02": ROOT / "surfaces/Pb111/H2_candidates/shared_bridges",
    "X01": ROOT / "surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_2p5",
    "X03": ROOT / "surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_6p0",
    "M2H07": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193",
    "X16": (
        ROOT
        / "surfaces/Pb111/PbH2_extraction/separation/"
        "fixed_mu/m0p1193/plus_1p5"
    ),
}


# ---------------------------------------------------------------------
# X17-X20: surface-two-H reference continuations
# ---------------------------------------------------------------------

GO_JOBS = [
    {
        "job_id": "X17",
        "parent_id": "M2H01",
        "target_mu_Ha": -0.11195,
        "approx_U_RHE_V_at_pH7": -1.2,
        "initial_state": "surface_2H_separated",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/separated/m0p11195",
        "job_name": "H2sep_mu11195",
        "purpose": (
            "Test whether the separated two-H surface basin remains a "
            "physical local minimum at target-mu = -0.11195 Ha."
        ),
    },
    {
        "job_id": "X18",
        "parent_id": "M2H04",
        "target_mu_Ha": -0.11195,
        "approx_U_RHE_V_at_pH7": -1.2,
        "initial_state": "surface_2H_same_Pb",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/same_Pb/m0p11195",
        "job_name": "H2same_mu11195",
        "purpose": (
            "Test whether the same-Pb two-H surface basin remains a "
            "physical local minimum at target-mu = -0.11195 Ha."
        ),
    },
    {
        "job_id": "X19",
        "parent_id": "M2H05",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "initial_state": "surface_2H_separated",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/separated/m0p09725",
        "job_name": "H2sep_mu09725",
        "purpose": (
            "Continue a chemically valid separated two-H surface state "
            "to target-mu = -0.09725 Ha."
        ),
    },
    {
        "job_id": "X20",
        "parent_id": "M2H04",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "initial_state": "surface_2H_same_Pb",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/same_Pb/m0p09725",
        "job_name": "H2same_mu09725",
        "purpose": (
            "Test whether the same-Pb two-H surface basin survives at "
            "target-mu = -0.09725 Ha."
        ),
    },
]


# ---------------------------------------------------------------------
# N05-N07: Pb(H)2 / PbH2 extraction NEBs
# ---------------------------------------------------------------------

NEB_JOBS = [
    {
        "job_id": "N05",
        "family": "neutral_neb",
        "initial_id": "HH02",
        "final_id": "X01",
        "target_mu_Ha": None,
        "approx_U_RHE_V_at_pH7": None,
        "energy_key": "F",
        "relpath": "kinetics/Pb111/neutral/PbH2_surface_to_partially_extracted",
        "job_name": "NEB_PbH2_part_neut",
        "purpose": (
            "Neutral first-stage NEB from same-Pb surface Pb(H)2 to the "
            "partially extracted Pb(H)2 minimum."
        ),
    },
    {
        "job_id": "N06",
        "family": "neutral_neb",
        "initial_id": "X01",
        "final_id": "X03",
        "target_mu_Ha": None,
        "approx_U_RHE_V_at_pH7": None,
        "energy_key": "F",
        "relpath": "kinetics/Pb111/neutral/PbH2_partially_extracted_to_detached",
        "job_name": "NEB_PbH2_det_neut",
        "purpose": (
            "Neutral first-stage NEB from partially extracted Pb(H)2 to "
            "detached PbH2 above a Pb vacancy."
        ),
    },
    {
        "job_id": "N07",
        "family": "fixed_mu_neb",
        "initial_id": "M2H07",
        "final_id": "X16",
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "energy_key": "G",
        "relpath": (
            "kinetics/Pb111/fixed_mu/m0p1193/"
            "PbH2_strongly_lifted_to_detached"
        ),
        "job_name": "NEB_PbH2_det_mu1193",
        "purpose": (
            "Fixed-mu first-stage NEB from strongly lifted surface Pb(H)2 "
            "to asymptotically detached PbH2 plus a Pb vacancy."
        ),
    },
]


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def replace_regex_once(text, pattern, replacement, description, flags=0):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=flags)

    if n != 1:
        raise RuntimeError(
            f"Could not uniquely replace {description}; matched {n} times."
        )

    return new_text


def constraint_signature(atoms):
    return [repr(constraint) for constraint in atoms.constraints]


def latest_matching(directory, pattern):
    candidates = [path for path in directory.glob(pattern) if path.is_file()]

    if not candidates:
        raise FileNotFoundError(
            f"No file matching {pattern!r} found in {directory}"
        )

    return max(candidates, key=lambda path: path.stat().st_mtime)


def validate_endpoint_pair(initial_dir, final_dir):
    initial = read(initial_dir / "CONTCAR", format="vasp")
    final = read(final_dir / "CONTCAR", format="vasp")

    initial.set_pbc((True, True, False))
    final.set_pbc((True, True, False))

    if len(initial) != len(final):
        raise ValueError(
            f"Endpoint atom counts differ: {initial_dir} vs {final_dir}"
        )

    if initial.get_chemical_symbols() != final.get_chemical_symbols():
        raise ValueError(
            f"Endpoint atom species/order differ: {initial_dir} vs {final_dir}"
        )

    if not np.allclose(
        initial.cell.array,
        final.cell.array,
        atol=1e-8,
        rtol=0.0,
    ):
        raise ValueError(
            f"Endpoint cells differ: {initial_dir} vs {final_dir}"
        )

    if constraint_signature(initial) != constraint_signature(final):
        raise ValueError(
            f"Endpoint constraints differ: {initial_dir} vs {final_dir}"
        )


def make_go_text(template, target_mu):
    if re.search(r"(?m)^\s*target-mu\s+", template):
        raise RuntimeError("Canonical go.py unexpectedly already contains target-mu")

    marker = "fluid-anion F- 0.5\n"

    if marker not in template:
        raise RuntimeError("Could not locate electrolyte block in canonical go.py")

    text = template.replace(
        marker,
        marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    text = replace_regex_once(
        text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "GO FIRE maxstep",
    )

    return text


def make_neb_text(template, energy_key, target_mu):
    text = template

    old_energy_function = re.compile(
        r"def _read_endpoint_energy\(s\):.*?"
        r"(?=\ndef _read_endpoint_forces)",
        flags=re.DOTALL,
    )

    new_energy_function = f'''def _read_endpoint_energy(s):
    f = next(n for n in os.listdir(s) if n.endswith('Ecomponents'))
    label = "{energy_key}"
    with open(os.path.join(s, f)) as fh:
        lines = [ln.strip() for ln in fh if ln.strip()]
    for line in reversed(lines):
        fields = line.split()
        if len(fields) >= 3 and fields[0] == label and fields[1] == '=':
            return float(fields[2]) * Hartree
    raise RuntimeError(f"Could not find {{label}} in {{os.path.join(s, f)}}")
'''

    text, n = old_energy_function.subn(
        new_energy_function,
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not replace NEB endpoint-energy parser")

    text = replace_regex_once(
        text,
        r"^NIMAGES\s*=\s*\d+\s*$",
        f"NIMAGES = {NIMAGES}",
        "NEB image count",
        flags=re.MULTILINE,
    )

    if target_mu is not None:
        if re.search(r"(?m)^\s*target-mu\s+", text):
            raise RuntimeError("Canonical neb.py unexpectedly already contains target-mu")

        marker = "fluid-anion F- 0.5\n"

        if marker not in text:
            raise RuntimeError("Could not locate electrolyte block in canonical neb.py")

        text = text.replace(
            marker,
            marker + f"target-mu {target_mu:.5f}\n",
            1,
        )

    text = replace_regex_once(
        text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "NEB FIRE maxstep",
    )

    text = replace_regex_once(
        text,
        r"opt\.run\(fmax\s*=\s*[0-9.]+,\s*steps\s*=\s*\d+\)",
        "opt.run(fmax=0.08, steps=150)",
        "NEB optimizer settings",
    )

    if "climb=False" not in text:
        raise RuntimeError("Expected canonical neb.py to contain climb=False")

    return text


def make_submit_text(template, job_name, neb=False):
    text = replace_regex_once(
        template,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "Slurm job name",
        flags=re.MULTILINE,
    )

    if neb:
        text = replace_regex_once(
            text,
            r"^#SBATCH -q .*?$",
            "#SBATCH -q regular",
            "NEB QoS",
            flags=re.MULTILINE,
        )

        text = replace_regex_once(
            text,
            r"^#SBATCH --time .*?$",
            "#SBATCH --time 08:00:00",
            "NEB walltime",
            flags=re.MULTILINE,
        )

    return text


def copy_endpoint_files(source_dir, destination_dir):
    destination_dir.mkdir(parents=True)

    shutil.copy2(
        source_dir / "CONTCAR",
        destination_dir / "CONTCAR",
    )

    ecomponents = latest_matching(source_dir, "*Ecomponents")
    force = latest_matching(source_dir, "*force")

    shutil.copy2(
        ecomponents,
        destination_dir / "endpoint.Ecomponents",
    )

    shutil.copy2(
        force,
        destination_dir / "endpoint.force",
    )


# ---------------------------------------------------------------------
# Build deterministic Git-tracked campaign manifest before writing jobs
# ---------------------------------------------------------------------

manifest_jobs = []

for job in GO_JOBS:
    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "type": "geometry_optimization",
            "directory": job["relpath"],
            "source_job_id": job["parent_id"],
            "target_mu_Ha": job["target_mu_Ha"],
            "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
            "initial_state": job["initial_state"],
            "purpose": job["purpose"],
        }
    )

for job in NEB_JOBS:
    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "type": "NEB",
            "family": job["family"],
            "directory": job["relpath"],
            "initial_job_id": job["initial_id"],
            "final_job_id": job["final_id"],
            "target_mu_Ha": job["target_mu_Ha"],
            "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
            "endpoint_energy_key": job["energy_key"],
            "interpolation": "ASE IDPP",
            "mic": True,
            "stage": "initial_band_relaxation",
            "purpose": job["purpose"],
        }
    )

manifest = {
    "wave": 6,
    "name": "surface 2H potential references and PbH2 extraction NEBs",
    "manifest_role": "campaign_definition",
    "scientific_questions": [
        (
            "Determine whether chemically valid surface two-H minima survive "
            "at target-mu = -0.11195 and -0.09725 Ha."
        ),
        (
            "Resolve the neutral Pb(H)2 extraction pathway into surface, "
            "partially extracted, and detached states."
        ),
        (
            "Calculate the initial non-climbing grand-canonical PbH2 extraction "
            "band at target-mu = -0.1193 Ha."
        ),
    ],
    "potential_mapping_note": (
        "Approximate RHE potentials use the historical project calibration. "
        "target_mu_Ha is the authoritative calculation variable."
    ),
    "interpretation_note": (
        "State labels for X17-X20 describe intended starting basins only. "
        "Final relaxed structures must be classified independently, and "
        "proton-well states must not be treated as physical references."
    ),
    "jobs": manifest_jobs,
}


# ---------------------------------------------------------------------
# Preflight: perform all inexpensive validation before writing jobs
# ---------------------------------------------------------------------

required_files = [
    GO_TEMPLATE,
    SUBMIT_GO_TEMPLATE,
    NEB_TEMPLATE,
    SUBMIT_NEB_TEMPLATE,
    IDPP_SCRIPT,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required source file: {path}")

for job_id, path in PATHS.items():
    if not (path / "CONTCAR").is_file():
        raise FileNotFoundError(
            f"{job_id} CONTCAR missing: {path / 'CONTCAR'}"
        )

for job in NEB_JOBS:
    initial_dir = PATHS[job["initial_id"]]
    final_dir = PATHS[job["final_id"]]

    validate_endpoint_pair(initial_dir, final_dir)

    latest_matching(initial_dir, "*Ecomponents")
    latest_matching(initial_dir, "*force")
    latest_matching(final_dir, "*Ecomponents")
    latest_matching(final_dir, "*force")

target_dirs = [
    ROOT / job["relpath"]
    for job in [*GO_JOBS, *NEB_JOBS]
]

existing = [path for path in target_dirs if path.exists()]

if existing:
    raise FileExistsError(
        "Refusing to overwrite existing Wave 6 directories:\n"
        + "\n".join(str(path) for path in existing)
    )

if MANIFEST_PATH.exists():
    existing_manifest = json.loads(MANIFEST_PATH.read_text())

    if existing_manifest != manifest:
        raise RuntimeError(
            f"Existing Git-tracked manifest differs from this Wave 6 definition: "
            f"{MANIFEST_PATH}\nRefusing to overwrite it."
        )


# ---------------------------------------------------------------------
# Load canonical templates
# ---------------------------------------------------------------------

go_template = GO_TEMPLATE.read_text()
submit_go_template = SUBMIT_GO_TEMPLATE.read_text()
neb_template = NEB_TEMPLATE.read_text()
submit_neb_template = SUBMIT_NEB_TEMPLATE.read_text()


# ---------------------------------------------------------------------
# Create X17-X20
# ---------------------------------------------------------------------

for job in GO_JOBS:
    job_dir = ROOT / job["relpath"]
    parent_dir = PATHS[job["parent_id"]]

    parent_atoms = read(parent_dir / "CONTCAR", format="vasp")
    parent_atoms.set_pbc((True, True, False))

    job_dir.mkdir(parents=True)

    # Preserve the exact relaxed parent coordinates and selective-dynamics flags.
    shutil.copy2(
        parent_dir / "CONTCAR",
        job_dir / "POSCAR",
    )

    (job_dir / "go.py").write_text(
        make_go_text(
            go_template,
            job["target_mu_Ha"],
        )
    )

    (job_dir / "submit.sh").write_text(
        make_submit_text(
            submit_go_template,
            job["job_name"],
            neb=False,
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "fixed_mu_2H",
        "initial_state": job["initial_state"],
        "state_label_scope": "initial_intent",
        "purpose": job["purpose"],
        "source_job_id": job["parent_id"],
        "source_path": str(parent_dir.relative_to(ROOT)),
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [True, True, False],
        "n_atoms": len(parent_atoms),
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": 0.05,
    }

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    print(
        f"{job['job_id']:>4}  "
        f"source={job['parent_id']:<5}  "
        f"mu={job['target_mu_Ha']: .5f}  "
        f"{job_dir}"
    )


# ---------------------------------------------------------------------
# Create N05-N07
# ---------------------------------------------------------------------

for job in NEB_JOBS:
    neb_dir = ROOT / job["relpath"]
    initial_dir = PATHS[job["initial_id"]]
    final_dir = PATHS[job["final_id"]]

    neb_dir.mkdir(parents=True)

    copy_endpoint_files(
        initial_dir,
        neb_dir / "00",
    )

    copy_endpoint_files(
        final_dir,
        neb_dir / f"{NIMAGES + 1:02d}",
    )

    (neb_dir / "neb.py").write_text(
        make_neb_text(
            neb_template,
            energy_key=job["energy_key"],
            target_mu=job["target_mu_Ha"],
        )
    )

    (neb_dir / "submit.sh").write_text(
        make_submit_text(
            submit_neb_template,
            job["job_name"],
            neb=True,
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": job["family"],
        "stage": "initial_band_relaxation",
        "purpose": job["purpose"],
        "initial_job_id": job["initial_id"],
        "final_job_id": job["final_id"],
        "initial_endpoint": str(initial_dir.relative_to(ROOT)),
        "final_endpoint": str(final_dir.relative_to(ROOT)),
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "endpoint_energy_key": job["energy_key"],
        "n_intermediate_images": NIMAGES,
        "interpolation": "ASE IDPP",
        "mic": True,
        "climb": False,
        "neb_method": "improvedtangent",
        "fire_maxstep_A": 0.05,
        "fmax_threshold_eV_A": 0.08,
        "max_fire_steps": 150,
        "kpts": [4, 4, 1, "gamma"],
        "setup_script": str(IDPP_SCRIPT.relative_to(SOURCE_ROOT)),
    }

    (neb_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )


# ---------------------------------------------------------------------
# Initialize N05-N07 with canonical MIC-aware IDPP
# ---------------------------------------------------------------------

print()
print("IDPP initialization")
print("===================")

for job in NEB_JOBS:
    neb_dir = ROOT / job["relpath"]

    print()
    print(f"{job['job_id']}: {neb_dir}")
    print("-" * 80)

    subprocess.run(
        [
            sys.executable,
            str(IDPP_SCRIPT),
            "--dir",
            str(neb_dir),
            "--nimages",
            str(NIMAGES),
        ],
        check=True,
        text=True,
    )


# ---------------------------------------------------------------------
# Write / validate Git-tracked Wave 6 campaign manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():
    print()
    print(f"Manifest already exists and matches: {MANIFEST_PATH}")
else:
    MANIFEST_PATH.write_text(
        json.dumps(manifest, indent=2) + "\n"
    )

    print()
    print(f"Wrote manifest: {MANIFEST_PATH}")


print()
print(f"Created {len(manifest_jobs)} Wave 6 jobs.")

for job in manifest_jobs:
    print(
        f"{job['job_id']:>4}  "
        f"{job['type']:<22}  "
        f"{job['directory']}"
    )

In [ ]:
# Purpose: Generate Wave 7 explicit-water proton-well rescue calculations.
#
# Input:
#   M2H07 CONTCAR: strongly lifted same-Pb Pb(H)2 state at target-mu = -0.1193 Ha
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   X21: hydride-oriented 2H2O shell, target-mu = -0.11930 Ha
#   X22: hydride-oriented 2H2O shell, target-mu = -0.09725 Ha
#   X23: proton-oriented 2H2O shell, target-mu = -0.11930 Ha
#   X24: proton-oriented 2H2O shell, target-mu = -0.09725 Ha
#   Pb_LAR_source/campaigns/wave7_proton_well_rescue.json
#
# Scientific question:
#   Test whether explicit first-shell water prevents the CANDLE proton-well
#   artifact by maintaining a physically excluded cavity around Pb-bound H.
#
# Water motifs:
#   hydride-oriented:
#       Pb-H ... H-O-H
#       one water H donates toward each Pb-bound H
#       initial H_water ... H_Pb contact = 1.75 Å
#
#   proton-oriented:
#       Pb-H ... O(H)2
#       water O points toward each Pb-bound H
#       initial O_water ... H_Pb contact = 1.90 Å
#
# Notes:
#   target-mu is the authoritative electrochemical control variable.
#   Approximate RHE potentials are historical metadata only.
#   State labels describe INITIAL intended structures. Final relaxed structures
#   must be classified independently for proton wells, intact water, Pb-H
#   bonding, and possible H2 formation.
#   These methodological jobs additionally request Vcavity output.
#   This cell refuses to overwrite existing calculation directories.

from pathlib import Path
import json
import re

import numpy as np
from ase import Atoms
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

PARENT_DIR = ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193"
PARENT_PATH = PARENT_DIR / "CONTCAR"

MANIFEST_DIR = SOURCE_ROOT / "campaigns"
MANIFEST_PATH = MANIFEST_DIR / "wave7_proton_well_rescue.json"

OH_BOND_A = 0.970
HOH_ANGLE_DEG = 104.5

# Hydride-oriented motif: H_water ... H_Pb
HYDRIDE_HH_CONTACT_A = 1.75

# Proton-oriented motif: O_water ... H_Pb
PROTON_OH_CONTACT_A = 1.90

GO_MAXSTEP_A = 0.03


JOBS = [
    {
        "job_id": "X21",
        "orientation": "hydride_oriented",
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "job_name": "PWres_hyd_m1193",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1193",
    },
    {
        "job_id": "X22",
        "orientation": "hydride_oriented",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "job_name": "PWres_hyd_m09725",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
    },
    {
        "job_id": "X23",
        "orientation": "proton_oriented",
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "job_name": "PWres_prot_m1193",
        "relpath": "surfaces/Pb111/proton_well_rescue/proton_oriented/m0p1193",
    },
    {
        "job_id": "X24",
        "orientation": "proton_oriented",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "job_name": "PWres_prot_m09725",
        "relpath": "surfaces/Pb111/proton_well_rescue/proton_oriented/m0p09725",
    },
]


# ---------------------------------------------------------------------
# Geometry helpers
# ---------------------------------------------------------------------

def unit(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)

    if norm < 1e-10:
        raise ValueError("Cannot normalize near-zero vector.")

    return vector / norm


def perpendicular_component(vector, axis):
    vector = np.asarray(vector, dtype=float)
    axis = unit(axis)

    result = vector - np.dot(vector, axis) * axis

    if np.linalg.norm(result) > 1e-6:
        return unit(result)

    candidates = [
        np.array([1.0, 0.0, 0.0]),
        np.array([0.0, 1.0, 0.0]),
        np.array([0.0, 0.0, 1.0]),
    ]

    candidate = min(candidates, key=lambda v: abs(np.dot(v, axis)))
    return unit(candidate - np.dot(candidate, axis) * axis)


def surface_z(atoms):
    symbols = np.array(atoms.get_chemical_symbols())
    pb_indices = np.where(symbols == "Pb")[0]

    if len(pb_indices) < 9:
        raise ValueError(f"Expected at least 9 Pb atoms, found {len(pb_indices)}")

    pb_z = atoms.positions[pb_indices, 2]
    return float(np.median(np.sort(pb_z)[-9:]))


def identify_pbh2(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(pb_indices) != 36:
        raise ValueError(f"M2H07 parent should contain 36 Pb atoms; found {len(pb_indices)}")

    if len(h_indices) != 2:
        raise ValueError(f"M2H07 parent should contain exactly 2 H atoms; found {len(h_indices)}")

    nearest_pb = []
    pb_h_distances = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(pb_idx), int(h_idx), mic=True)
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))

        nearest_pb.append(int(pb_indices[local]))
        pb_h_distances.append(float(distances[local]))

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two parent H atoms do not share one Pb: {nearest_pb}"
        )

    if max(pb_h_distances) > 2.4:
        raise ValueError(
            f"Parent does not look like intact Pb(H)2: Pb-H = {pb_h_distances}"
        )

    pb_idx = nearest_pb[0]
    pb_lift = float(atoms.positions[pb_idx, 2] - surface_z(atoms))

    if not 0.5 < pb_lift < 3.0:
        raise ValueError(
            "M2H07 does not resemble the expected strongly lifted Pb(H)2 state: "
            f"Pb lift = {pb_lift:.3f} Å"
        )

    return (
        pb_idx,
        [int(i) for i in h_indices],
        pb_h_distances,
        pb_lift,
    )


def local_geometry_vectors(atoms, pb_idx, h_indices, i):
    h_idx = h_indices[i]
    other_idx = h_indices[1 - i]

    pb_to_h = atoms.get_distance(
        pb_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    other_to_h = atoms.get_distance(
        other_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    outward = unit(pb_to_h)

    # Choose a direction perpendicular to Pb-H and pointing away from the
    # other Pb-bound H as much as possible.
    sideways = perpendicular_component(other_to_h, outward)

    return outward, sideways


def make_hydride_oriented_water(target_position, outward, sideways):
    """
    Construct Pb-H ... H-O-H.

    One water H is placed between the Pb-bound H and water O, so an O-H
    bond donates toward the hydride-like Pb-H.
    """
    theta = np.deg2rad(HOH_ANGLE_DEG)

    donor_h = target_position + HYDRIDE_HH_CONTACT_A * outward
    oxygen = donor_h + OH_BOND_A * outward

    # O -> donor H is -outward. Place the second O-H bond at the water angle.
    second_direction = (
        np.cos(theta) * (-outward)
        + np.sin(theta) * sideways
    )

    second_h = oxygen + OH_BOND_A * unit(second_direction)

    return Atoms(
        symbols=["O", "H", "H"],
        positions=[oxygen, donor_h, second_h],
    )


def make_proton_oriented_water(target_position, outward, sideways):
    """
    Construct Pb-H ... O(H)2.

    Water O points toward the Pb-bound H and both O-H bonds point generally
    away from the target H.
    """
    half_angle = np.deg2rad(HOH_ANGLE_DEG / 2.0)

    oxygen = target_position + PROTON_OH_CONTACT_A * outward

    h1_direction = (
        np.cos(half_angle) * outward
        + np.sin(half_angle) * sideways
    )

    h2_direction = (
        np.cos(half_angle) * outward
        - np.sin(half_angle) * sideways
    )

    h1 = oxygen + OH_BOND_A * unit(h1_direction)
    h2 = oxygen + OH_BOND_A * unit(h2_direction)

    return Atoms(
        symbols=["O", "H", "H"],
        positions=[oxygen, h1, h2],
    )


def add_two_waters(parent, orientation, pb_idx, h_indices):
    atoms = parent.copy()
    water_records = []

    for i, h_idx in enumerate(h_indices):
        outward, sideways = local_geometry_vectors(
            parent,
            pb_idx,
            h_indices,
            i,
        )

        target = parent.positions[h_idx].copy()

        if orientation == "hydride_oriented":
            water = make_hydride_oriented_water(
                target,
                outward,
                sideways,
            )

        elif orientation == "proton_oriented":
            water = make_proton_oriented_water(
                target,
                outward,
                sideways,
            )

        else:
            raise ValueError(f"Unknown orientation: {orientation}")

        water.set_cell(parent.cell)
        water.set_pbc(parent.pbc)

        first_new_index = len(atoms)
        atoms += water

        water_records.append(
            {
                "target_parent_H_index": int(h_idx),
                "water_initial_indices_before_go_sort": [
                    first_new_index,
                    first_new_index + 1,
                    first_new_index + 2,
                ],
            }
        )

    atoms.set_pbc((True, True, False))

    return atoms, water_records


def validate_rescue_geometry(
    atoms,
    pb_idx,
    parent_h_indices,
    water_records,
    orientation,
):
    symbols = np.array(atoms.get_chemical_symbols())

    n_pb = int(np.sum(symbols == "Pb"))
    n_o = int(np.sum(symbols == "O"))
    n_h = int(np.sum(symbols == "H"))

    if (n_pb, n_o, n_h) != (36, 2, 6):
        raise ValueError(
            f"Unexpected composition Pb/O/H = {n_pb}/{n_o}/{n_h}; expected 36/2/6."
        )

    if len(water_records) != 2:
        raise ValueError(f"Expected exactly two explicit-water records; found {len(water_records)}")

    water_groups = [
        record["water_initial_indices_before_go_sort"]
        for record in water_records
    ]

    contact_distances = []
    water_geometry = []

    for i, group in enumerate(water_groups):
        if len(group) != 3:
            raise ValueError(f"Water {i + 1} does not contain exactly three recorded atoms.")

        o_idx, wh1_idx, wh2_idx = group
        target_h = parent_h_indices[i]

        if symbols[o_idx] != "O" or symbols[wh1_idx] != "H" or symbols[wh2_idx] != "H":
            raise ValueError(
                f"Unexpected species ordering for water {i + 1}: "
                f"{symbols[o_idx]}, {symbols[wh1_idx]}, {symbols[wh2_idx]}"
            )

        oh1 = float(atoms.get_distance(o_idx, wh1_idx, mic=True))
        oh2 = float(atoms.get_distance(o_idx, wh2_idx, mic=True))

        v1 = atoms.get_distance(o_idx, wh1_idx, mic=True, vector=True)
        v2 = atoms.get_distance(o_idx, wh2_idx, mic=True, vector=True)

        angle = float(
            np.degrees(
                np.arccos(
                    np.clip(
                        np.dot(unit(v1), unit(v2)),
                        -1.0,
                        1.0,
                    )
                )
            )
        )

        if not (0.94 <= oh1 <= 1.00 and 0.94 <= oh2 <= 1.00):
            raise ValueError(
                f"Water {i + 1} O-H geometry is bad: {oh1:.3f}, {oh2:.3f} Å"
            )

        if not 102.0 <= angle <= 107.0:
            raise ValueError(
                f"Water {i + 1} H-O-H angle is bad: {angle:.2f} deg"
            )

        if orientation == "hydride_oriented":
            contact = min(
                atoms.get_distance(target_h, wh1_idx, mic=True),
                atoms.get_distance(target_h, wh2_idx, mic=True),
            )
            expected_contact = HYDRIDE_HH_CONTACT_A

        else:
            contact = atoms.get_distance(target_h, o_idx, mic=True)
            expected_contact = PROTON_OH_CONTACT_A

        contact = float(contact)

        if abs(contact - expected_contact) > 0.05:
            raise ValueError(
                f"Water {i + 1} target contact is {contact:.3f} Å; "
                f"expected approximately {expected_contact:.3f} Å."
            )

        contact_distances.append(contact)

        water_geometry.append(
            {
                "water_number": i + 1,
                "OH_distances_A": [oh1, oh2],
                "HOH_angle_deg": angle,
                "target_contact_A": contact,
            }
        )

    interwater_distances = [
        atoms.get_distance(i, j, mic=True)
        for i in water_groups[0]
        for j in water_groups[1]
    ]

    min_interwater = float(min(interwater_distances))

    if min_interwater < 1.35:
        raise ValueError(
            "The two explicit waters overlap: minimum cross-water "
            f"distance = {min_interwater:.3f} Å"
        )

    max_z = float(np.max(atoms.positions[:, 2]))
    top_gap = float(atoms.cell[2, 2] - max_z)

    if top_gap < 4.0:
        raise ValueError(
            f"Only {top_gap:.2f} Å remains above explicit atoms."
        )

    pb_h = [
        float(atoms.get_distance(pb_idx, h_idx, mic=True))
        for h_idx in parent_h_indices
    ]

    hh = float(
        atoms.get_distance(
            parent_h_indices[0],
            parent_h_indices[1],
            mic=True,
        )
    )

    return {
        "n_Pb": n_pb,
        "n_O": n_o,
        "n_H": n_h,
        "initial_PbH_distances_A": pb_h,
        "initial_HH_distance_A": hh,
        "initial_water_target_contacts_A": contact_distances,
        "initial_water_geometry": water_geometry,
        "initial_min_interwater_distance_A": min_interwater,
        "initial_top_gap_A": top_gap,
    }


# ---------------------------------------------------------------------
# JDFTx / Slurm text construction
# ---------------------------------------------------------------------

def make_go_text(template, target_mu):
    text = template

    if re.search(r"(?m)^\s*target-mu\s+", text):
        raise RuntimeError(
            "Canonical go.py already contains target-mu; refusing ambiguous edit."
        )

    fluid_marker = "fluid-anion F- 0.5\n"

    if text.count(fluid_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid-anion command in canonical go.py."
        )

    text = text.replace(
        fluid_marker,
        fluid_marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    if re.search(r"(?m)^dump Ionic .*Vcavity", text):
        raise RuntimeError(
            "Canonical go.py unexpectedly already requests Vcavity."
        )

    cavity_marker = "dump Ionic BoundCharge VfluidTot FluidDensity\n"

    if text.count(cavity_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid dump command in canonical go.py."
        )

    text = text.replace(
        cavity_marker,
        cavity_marker + "dump Ionic Vcavity\n",
        1,
    )

    text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={GO_MAXSTEP_A:.2f}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep.")

    return text


def make_submit_text(template, job_name):
    text, n = re.subn(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        template,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name.")

    return text


# ---------------------------------------------------------------------
# Deterministic Git-tracked campaign definition
# ---------------------------------------------------------------------

manifest_jobs = [
    {
        "job_id": job["job_id"],
        "type": "geometry_optimization",
        "directory": job["relpath"],
        "orientation": job["orientation"],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "source_job_id": "M2H07",
        "initial_state": "strongly_lifted_PbH2_plus_2H2O",
    }
    for job in JOBS
]

manifest = {
    "wave": 7,
    "name": "explicit-water proton-well rescue pilot",
    "manifest_role": "campaign_definition",
    "scientific_question": (
        "Can a chemically bound strongly lifted Pb(H)2 state survive at "
        "cathodic target-mu when two explicit first-shell water molecules "
        "prevent the implicit-solvent cavity from approaching bare H directly?"
    ),
    "strategy": (
        "Compare hydride-oriented and proton-oriented water motifs at a "
        "known-good control potential (-0.11930 Ha) and a potential where "
        "dry CANDLE calculations exhibit the proton-well artifact (-0.09725 Ha)."
    ),
    "construction": {
        "explicit_water_count": 2,
        "OH_bond_A": OH_BOND_A,
        "HOH_angle_deg": HOH_ANGLE_DEG,
        "hydride_Hwater_contact_A": HYDRIDE_HH_CONTACT_A,
        "proton_Owater_contact_A": PROTON_OH_CONTACT_A,
    },
    "potential_mapping_note": (
        "Approximate RHE potentials use the historical project calibration. "
        "target_mu_Ha is the authoritative calculation variable."
    ),
    "interpretation_note": (
        "Initial-state labels do not establish the final relaxed chemistry. "
        "Final structures must be independently classified for intact water, "
        "Pb-H bonding, proton-well formation, and H2 formation."
    ),
    "jobs": manifest_jobs,
}


# ---------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------

for path in [GO_TEMPLATE, SUBMIT_TEMPLATE, PARENT_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")

target_dirs = [ROOT / job["relpath"] for job in JOBS]
existing = [path for path in target_dirs if path.exists()]

if existing:
    raise FileExistsError(
        "Refusing to overwrite existing Wave 7 directories:\n"
        + "\n".join(str(path) for path in existing)
    )

if MANIFEST_PATH.exists():
    existing_manifest = json.loads(MANIFEST_PATH.read_text())

    if existing_manifest != manifest:
        raise RuntimeError(
            f"Existing Git-tracked manifest differs from this Wave 7 definition: "
            f"{MANIFEST_PATH}\nRefusing to overwrite it."
        )


# ---------------------------------------------------------------------
# Load and validate M2H07
# ---------------------------------------------------------------------

parent = read(PARENT_PATH, format="vasp")
parent.set_pbc((True, True, False))

pb_idx, hydride_indices, parent_pb_h, parent_pb_lift = identify_pbh2(parent)

parent_hh = float(
    parent.get_distance(
        hydride_indices[0],
        hydride_indices[1],
        mic=True,
    )
)

print("Wave 7 parent")
print("=============")
print("Parent:           M2H07")
print(f"Common Pb index:  {pb_idx}")
print(f"Hydride indices:  {hydride_indices}")
print(
    "Pb-H distances:  "
    + ", ".join(f"{x:.4f} Å" for x in parent_pb_h)
)
print(f"H-H distance:     {parent_hh:.4f} Å")
print(f"Pb lift:          {parent_pb_lift:.4f} Å")
print()


# ---------------------------------------------------------------------
# Construct and validate both explicit-water starting motifs before
# modifying the calculation tree
# ---------------------------------------------------------------------

orientation_structures = {}
orientation_records = {}
orientation_metrics = {}

for orientation in ["hydride_oriented", "proton_oriented"]:
    atoms, water_records = add_two_waters(
        parent,
        orientation,
        pb_idx,
        hydride_indices,
    )

    metrics = validate_rescue_geometry(
        atoms,
        pb_idx,
        hydride_indices,
        water_records,
        orientation,
    )

    orientation_structures[orientation] = atoms
    orientation_records[orientation] = water_records
    orientation_metrics[orientation] = metrics


# Validate template transformations before creating any job directories.
go_template = GO_TEMPLATE.read_text()
submit_template = SUBMIT_TEMPLATE.read_text()

go_texts = {}
submit_texts = {}

for job in JOBS:
    go_texts[job["job_id"]] = make_go_text(
        go_template,
        job["target_mu_Ha"],
    )

    submit_texts[job["job_id"]] = make_submit_text(
        submit_template,
        job["job_name"],
    )


# ---------------------------------------------------------------------
# Create X21-X24
# ---------------------------------------------------------------------

for job in JOBS:
    orientation = job["orientation"]
    atoms = orientation_structures[orientation].copy()
    metrics = orientation_metrics[orientation]

    job_dir = ROOT / job["relpath"]
    job_dir.mkdir(parents=True)

    write(
        job_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    # Human-readable construction snapshot.
    write(
        job_dir / "initial.xyz",
        atoms,
        format="extxyz",
    )

    (job_dir / "go.py").write_text(go_texts[job["job_id"]])
    (job_dir / "submit.sh").write_text(submit_texts[job["job_id"]])

    metadata = {
        "job_id": job["job_id"],
        "family": "proton_well_rescue",
        "initial_state": "strongly_lifted_PbH2_plus_2H2O",
        "state_label_scope": "initial_intent",
        "purpose": (
            "Test whether two explicit first-shell water molecules prevent "
            "the CANDLE proton-well artifact for strongly lifted surface "
            "Pb(H)2 at fixed electron chemical potential."
        ),
        "source_job_id": "M2H07",
        "source_path": str(PARENT_DIR.relative_to(ROOT)),
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "explicit_water_count": 2,
        "water_orientation": orientation,
        "construction": {
            "OH_bond_A": OH_BOND_A,
            "HOH_angle_deg": HOH_ANGLE_DEG,
            "hydride_Hwater_contact_A": (
                HYDRIDE_HH_CONTACT_A
                if orientation == "hydride_oriented"
                else None
            ),
            "proton_Owater_contact_A": (
                PROTON_OH_CONTACT_A
                if orientation == "proton_oriented"
                else None
            ),
            "common_parent_Pb_index_before_go_sort": pb_idx,
            "parent_H_indices_before_go_sort": hydride_indices,
            "water_records_before_go_sort": orientation_records[orientation],
        },
        "initial_geometry_metrics": metrics,
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [True, True, False],
        "n_atoms": len(atoms),
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": GO_MAXSTEP_A,
        "requested_fluid_diagnostics": [
            "FluidDensity",
            "BoundCharge",
            "VfluidTot",
            "Vcavity",
        ],
        "potential_mapping_note": (
            "approx_U_RHE_V_at_pH7 uses the historical project calibration; "
            "target_mu_Ha is the authoritative calculation variable."
        ),
    }

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )


# ---------------------------------------------------------------------
# Write / validate Git-tracked Wave 7 manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():
    print(f"Manifest already exists and matches: {MANIFEST_PATH}")
else:
    MANIFEST_PATH.write_text(
        json.dumps(manifest, indent=2) + "\n"
    )
    print(f"Wrote manifest: {MANIFEST_PATH}")


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print()
print("Wave 7 starting-geometry checks")
print("===============================")

for orientation in ["hydride_oriented", "proton_oriented"]:
    metrics = orientation_metrics[orientation]

    print()
    print(orientation)
    print("-" * len(orientation))

    print(
        "Pb-H:             "
        + ", ".join(
            f"{x:.4f} Å"
            for x in metrics["initial_PbH_distances_A"]
        )
    )

    print(
        f"PbH2 H-H:          "
        f"{metrics['initial_HH_distance_A']:.4f} Å"
    )

    print(
        "water contacts:   "
        + ", ".join(
            f"{x:.4f} Å"
            for x in metrics["initial_water_target_contacts_A"]
        )
    )

    print(
        f"min water-water:   "
        f"{metrics['initial_min_interwater_distance_A']:.4f} Å"
    )

    print(
        f"top vacuum gap:    "
        f"{metrics['initial_top_gap_A']:.4f} Å"
    )

print()
print(f"Created {len(JOBS)} Wave 7 jobs.")

for job in manifest_jobs:
    print(
        f"{job['job_id']:>4}  "
        f"mu={job['target_mu_Ha']: .5f}  "
        f"{job['orientation']:<18}  "
        f"{job['directory']}"
    )

In [ ]:
# Purpose: Generate Wave 8 continuation of the successful hydride-oriented
# explicit-water Pb(H)2 branch across fixed electron chemical potential.
#
# Inputs:
#   X21 CONTCAR: hydride-oriented Pb(H)2 + 2H2O at target-mu = -0.11930 Ha
#   X22 CONTCAR: hydride-oriented Pb(H)2 + 2H2O at target-mu = -0.09725 Ha
#   analysis/surface_results.csv
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   X25: target-mu = -0.11195 Ha, continued from X21
#   X26: target-mu = -0.10460 Ha, continued from X22
#   X27: target-mu = -0.08990 Ha, continued from X22
#   Pb_LAR_source/campaigns/wave8_explicit_water_potential_continuation.json
#
# Continuation strategy:
#   Use the nearest available converged physical hydride-oriented explicit-water
#   state as the starting geometry.
#
# Notes:
#   target-mu is the authoritative electrochemical control variable.
#   Approximate RHE potentials are historical metadata only.
#   X21/X22 must be both converged and physically valid before being propagated.
#   Final X25-X27 structures must again be independently classified.
#   Existing calculation directories are never overwritten.

from pathlib import Path
import csv
import json
import re

import numpy as np
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

RESULTS_CSV = ROOT / "analysis" / "surface_results.csv"

MANIFEST_DIR = SOURCE_ROOT / "campaigns"
MANIFEST_PATH = MANIFEST_DIR / "wave8_explicit_water_potential_continuation.json"

GO_MAXSTEP_A = 0.03
WATER_OH_CUTOFF_A = 1.25
MAX_BOUND_PBH_A = 2.60
MIN_NON_H2_HH_A = 1.20


JOBS = [
    {
        "job_id": "X25",
        "source_job_id": "X21",
        "source_target_mu_Ha": -0.11930,
        "source_relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1193",
        "target_mu_Ha": -0.11195,
        "approx_U_RHE_V_at_pH7": -1.2,
        "job_name": "PWres_hyd_m11195",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p11195",
    },
    {
        "job_id": "X26",
        "source_job_id": "X22",
        "source_target_mu_Ha": -0.09725,
        "source_relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
        "target_mu_Ha": -0.10460,
        "approx_U_RHE_V_at_pH7": -1.4,
        "job_name": "PWres_hyd_m1046",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1046",
    },
    {
        "job_id": "X27",
        "source_job_id": "X22",
        "source_target_mu_Ha": -0.09725,
        "source_relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
        "target_mu_Ha": -0.08990,
        "approx_U_RHE_V_at_pH7": -1.8,
        "job_name": "PWres_hyd_m0899",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p0899",
    },
]


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def load_harvest_status(csv_path):
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Missing harvested results: {csv_path}\n"
            "Run harvest_surface_results.py after X21/X22 finish."
        )

    rows = {}

    with csv_path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            rows[row["job_id"]] = row

    return rows


def make_go_text(template, target_mu):
    text = template

    if re.search(r"(?m)^\s*target-mu\s+", text):
        raise RuntimeError(
            "Canonical go.py unexpectedly already contains target-mu."
        )

    fluid_marker = "fluid-anion F- 0.5\n"

    if text.count(fluid_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid-anion command in canonical go.py."
        )

    text = text.replace(
        fluid_marker,
        fluid_marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    cavity_marker = "dump Ionic BoundCharge VfluidTot FluidDensity\n"

    if re.search(r"(?m)^dump Ionic .*Vcavity", text):
        raise RuntimeError(
            "Canonical go.py unexpectedly already requests Vcavity."
        )

    if text.count(cavity_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid dump command in canonical go.py."
        )

    text = text.replace(
        cavity_marker,
        cavity_marker + "dump Ionic Vcavity\n",
        1,
    )

    text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={GO_MAXSTEP_A:.2f}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep.")

    return text


def make_submit_text(template, job_name):
    text, n = re.subn(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        template,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name.")

    return text


def validate_hydrated_pbh2(atoms):
    """
    Validate that a source state contains:
      - 36 Pb
      - 2 intact H2O molecules
      - 2 remaining Pb-bound H atoms
      - no short H-H bond consistent with H2
      - both Pb-bound H atoms associated with the same Pb

    This is a source-parent sanity check, not a general final-state classifier.
    """
    atoms = atoms.copy()
    atoms.set_pbc((True, True, False))

    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    o_indices = np.where(symbols == "O")[0]
    h_indices = np.where(symbols == "H")[0]

    if (len(pb_indices), len(o_indices), len(h_indices)) != (36, 2, 6):
        raise ValueError(
            "Expected hydrated Pb(H)2 composition Pb/O/H = 36/2/6; "
            f"found {len(pb_indices)}/{len(o_indices)}/{len(h_indices)}."
        )

    water_h_sets = []

    for o_idx in o_indices:
        nearby_h = [
            int(h_idx)
            for h_idx in h_indices
            if atoms.get_distance(int(o_idx), int(h_idx), mic=True) <= WATER_OH_CUTOFF_A
        ]

        if len(nearby_h) != 2:
            raise ValueError(
                f"O atom {o_idx} has {len(nearby_h)} H atoms within "
                f"{WATER_OH_CUTOFF_A:.2f} Å; expected 2."
            )

        water_h_sets.append(set(nearby_h))

    if water_h_sets[0] & water_h_sets[1]:
        raise ValueError(
            "The two O atoms share an H within the water O-H cutoff; "
            "explicit-water structure is not two intact H2O molecules."
        )

    water_h_indices = sorted(set().union(*water_h_sets))

    if len(water_h_indices) != 4:
        raise ValueError(
            f"Expected 4 unique water H atoms; found {water_h_indices}."
        )

    bound_h_indices = [
        int(h_idx)
        for h_idx in h_indices
        if int(h_idx) not in water_h_indices
    ]

    if len(bound_h_indices) != 2:
        raise ValueError(
            f"Expected 2 non-water H atoms; found {bound_h_indices}."
        )

    nearest_pb_indices = []
    pb_h_distances = []

    for h_idx in bound_h_indices:
        distances = np.array([
            atoms.get_distance(h_idx, int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))
        nearest_pb = int(pb_indices[local])
        distance = float(distances[local])

        if distance > MAX_BOUND_PBH_A:
            raise ValueError(
                f"Non-water H atom {h_idx} is not clearly Pb-bound: "
                f"nearest Pb-H = {distance:.3f} Å."
            )

        nearest_pb_indices.append(nearest_pb)
        pb_h_distances.append(distance)

    if nearest_pb_indices[0] != nearest_pb_indices[1]:
        raise ValueError(
            "The two non-water H atoms do not share one Pb: "
            f"{nearest_pb_indices}"
        )

    hh_distance = float(
        atoms.get_distance(
            bound_h_indices[0],
            bound_h_indices[1],
            mic=True,
        )
    )

    if hh_distance < MIN_NON_H2_HH_A:
        raise ValueError(
            f"The two non-water H atoms have H-H = {hh_distance:.3f} Å; "
            "this resembles H2 rather than intact Pb(H)2."
        )

    water_oh_distances = []

    for o_idx, water_h in zip(o_indices, water_h_sets):
        water_oh_distances.append([
            float(atoms.get_distance(int(o_idx), h_idx, mic=True))
            for h_idx in sorted(water_h)
        ])

    return {
        "water_H_indices": water_h_indices,
        "PbH_H_indices": bound_h_indices,
        "common_Pb_index": nearest_pb_indices[0],
        "PbH_distances_A": pb_h_distances,
        "PbH_HH_distance_A": hh_distance,
        "water_OH_distances_A": water_oh_distances,
    }


def validate_source_job(job, harvest):
    source_job_id = job["source_job_id"]
    source_dir = ROOT / job["source_relpath"]

    if source_job_id not in harvest:
        raise KeyError(
            f"{source_job_id} not found in {RESULTS_CSV}"
        )

    status = harvest[source_job_id]["status"]

    if status != "CONVERGED":
        raise RuntimeError(
            f"{source_job_id} is not converged: status={status}"
        )

    required = [
        source_dir / "CONTCAR",
        source_dir / "metadata.json",
    ]

    for path in required:
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing required source file: {path}"
            )

    source_metadata = json.loads(
        (source_dir / "metadata.json").read_text()
    )

    if source_metadata.get("job_id") != source_job_id:
        raise ValueError(
            f"{source_dir}/metadata.json identifies "
            f"{source_metadata.get('job_id')}, expected {source_job_id}."
        )

    if source_metadata.get("family") != "proton_well_rescue":
        raise ValueError(
            f"{source_job_id} has unexpected family: "
            f"{source_metadata.get('family')}"
        )

    if source_metadata.get("water_orientation") != "hydride_oriented":
        raise ValueError(
            f"{source_job_id} is not hydride-oriented: "
            f"{source_metadata.get('water_orientation')}"
        )

    source_mu = source_metadata.get("target_mu_Ha")

    if source_mu is None or not np.isclose(
        float(source_mu),
        job["source_target_mu_Ha"],
        atol=1e-8,
        rtol=0.0,
    ):
        raise ValueError(
            f"{source_job_id} has target-mu = {source_mu}; "
            f"expected {job['source_target_mu_Ha']} Ha."
        )

    atoms = read(source_dir / "CONTCAR", format="vasp")
    atoms.set_pbc((True, True, False))

    geometry = validate_hydrated_pbh2(atoms)

    return atoms, geometry


# ---------------------------------------------------------------------
# Deterministic Git-tracked campaign definition
# ---------------------------------------------------------------------

manifest_jobs = [
    {
        "job_id": job["job_id"],
        "type": "geometry_optimization",
        "directory": job["relpath"],
        "source_job_id": job["source_job_id"],
        "source_path": job["source_relpath"],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "initial_state": "hydride_oriented_PbH2_plus_2H2O",
    }
    for job in JOBS
]

manifest = {
    "wave": 8,
    "name": "hydride-oriented explicit-water potential continuation",
    "manifest_role": "campaign_definition",
    "scientific_question": (
        "Does the hydrated strongly lifted Pb(H)2 local minimum persist "
        "through the cathodic target-mu range where corresponding dry "
        "CANDLE calculations enter the proton-well artifact?"
    ),
    "reference_states": {
        "X21": {
            "target_mu_Ha": -0.11930,
            "role": "physical hydride-oriented explicit-water control",
        },
        "X22": {
            "target_mu_Ha": -0.09725,
            "role": "physical hydride-oriented explicit-water rescue in the dry proton-well regime",
        },
    },
    "continuation_strategy": (
        "Use the nearest available converged physical hydride-oriented "
        "explicit-water state as the starting geometry."
    ),
    "potential_mapping_note": (
        "Approximate RHE potentials use the historical project calibration. "
        "target_mu_Ha is the authoritative calculation variable."
    ),
    "interpretation_note": (
        "Source states must be converged and physically validated before "
        "continuation. Final X25-X27 structures must be classified independently."
    ),
    "jobs": manifest_jobs,
}


# ---------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------

for path in [GO_TEMPLATE, SUBMIT_TEMPLATE, RESULTS_CSV]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )

target_dirs = [
    ROOT / job["relpath"]
    for job in JOBS
]

existing = [
    path
    for path in target_dirs
    if path.exists()
]

if existing:
    raise FileExistsError(
        "Refusing to overwrite existing Wave 8 directories:\n"
        + "\n".join(str(path) for path in existing)
    )

if MANIFEST_PATH.exists():
    existing_manifest = json.loads(
        MANIFEST_PATH.read_text()
    )

    if existing_manifest != manifest:
        raise RuntimeError(
            f"Existing Git-tracked manifest differs from this Wave 8 definition: "
            f"{MANIFEST_PATH}\nRefusing to overwrite it."
        )

harvest = load_harvest_status(RESULTS_CSV)

source_atoms = {}
source_geometry = {}

for job in JOBS:
    source_job_id = job["source_job_id"]

    # X22 is used twice, so validate/read each unique source only once.
    if source_job_id in source_atoms:
        continue

    atoms, geometry = validate_source_job(
        job,
        harvest,
    )

    source_atoms[source_job_id] = atoms
    source_geometry[source_job_id] = geometry


# Validate canonical template transformations before creating directories.
go_template = GO_TEMPLATE.read_text()
submit_template = SUBMIT_TEMPLATE.read_text()

go_texts = {}
submit_texts = {}

for job in JOBS:
    go_texts[job["job_id"]] = make_go_text(
        go_template,
        job["target_mu_Ha"],
    )

    submit_texts[job["job_id"]] = make_submit_text(
        submit_template,
        job["job_name"],
    )


# ---------------------------------------------------------------------
# Report validated source states
# ---------------------------------------------------------------------

print("Wave 8 source-state validation")
print("==============================")

for source_job_id in sorted(source_geometry):
    geometry = source_geometry[source_job_id]

    print()
    print(source_job_id)
    print("-" * len(source_job_id))
    print(
        "Pb-H:   "
        + ", ".join(
            f"{value:.4f} Å"
            for value in geometry["PbH_distances_A"]
        )
    )
    print(
        f"H-H:    "
        f"{geometry['PbH_HH_distance_A']:.4f} Å"
    )
    print(
        f"Pb atom: {geometry['common_Pb_index']}"
    )


# ---------------------------------------------------------------------
# Create X25-X27
# ---------------------------------------------------------------------

for job in JOBS:
    source_job_id = job["source_job_id"]
    atoms = source_atoms[source_job_id].copy()

    target_dir = ROOT / job["relpath"]
    target_dir.mkdir(parents=True)

    write(
        target_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    write(
        target_dir / "initial.xyz",
        atoms,
        format="extxyz",
    )

    (target_dir / "go.py").write_text(
        go_texts[job["job_id"]]
    )

    (target_dir / "submit.sh").write_text(
        submit_texts[job["job_id"]]
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "proton_well_rescue",
        "initial_state": "hydride_oriented_PbH2_plus_2H2O",
        "state_label_scope": "initial_intent",
        "purpose": (
            "Continue the successful hydride-oriented explicit-water Pb(H)2 "
            "branch across fixed electron chemical potential to test whether "
            "first-shell water suppresses the CANDLE proton-well artifact."
        ),
        "source_job_id": source_job_id,
        "source_path": job["source_relpath"],
        "source_geometry_validation": source_geometry[source_job_id],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "explicit_water_count": 2,
        "water_orientation": "hydride_oriented",
        "continuation_strategy": (
            "Use the CONTCAR from the nearest available converged physical "
            "hydride-oriented explicit-water state."
        ),
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [True, True, False],
        "n_atoms": len(atoms),
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": GO_MAXSTEP_A,
        "requested_fluid_diagnostics": [
            "FluidDensity",
            "BoundCharge",
            "VfluidTot",
            "Vcavity",
        ],
        "potential_mapping_note": (
            "approx_U_RHE_V_at_pH7 uses the historical project calibration; "
            "target_mu_Ha is the authoritative calculation variable."
        ),
    }

    (target_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )


# ---------------------------------------------------------------------
# Write / validate Git-tracked Wave 8 campaign manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if MANIFEST_PATH.exists():
    print()
    print(
        f"Manifest already exists and matches: "
        f"{MANIFEST_PATH}"
    )
else:
    MANIFEST_PATH.write_text(
        json.dumps(manifest, indent=2) + "\n"
    )

    print()
    print(
        f"Wrote manifest: "
        f"{MANIFEST_PATH}"
    )


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print()
print("Wave 8 created")
print("==============")

for job in manifest_jobs:
    print(
        f"{job['job_id']:>4}  "
        f"mu={job['target_mu_Ha']: .5f}  "
        f"source={job['source_job_id']:<3}  "
        f"{job['directory']}"
    )

In [ ]:
# Purpose: Generate Wave 9 hydrated detached-PbH2 product states for balanced
# fixed-potential detachment thermodynamics.
#
# Inputs:
#   X06 CONTCAR: detached PbH2 + vacancy at target-mu = -0.11930 Ha
#   X14 CONTCAR: detached PbH2 + vacancy at target-mu = -0.09725 Ha
#   analysis/surface_results.csv
#   Pb_LAR_source/control/go.py
#   Pb_LAR_source/control/submit_go.sh
#
# Outputs:
#   X28: detached PbH2 + vacancy + 2H2O at target-mu = -0.11930 Ha
#   X29: detached PbH2 + vacancy + 2H2O at target-mu = -0.09725 Ha
#   Pb_LAR_source/campaigns/wave9_detached_PbH2_explicit_water_pilot.json
#
# Scientific purpose:
#   Add the same hydride-oriented first-shell water motif used for X21/X22 to
#   the detached PbH2 products. This creates equal-stoichiometry surface and
#   detached states for direct grand-free-energy comparison at fixed target-mu.
#
# Notes:
#   target-mu is the authoritative electrochemical control variable.
#   Approximate RHE potentials are historical metadata only.
#   Source states must be converged, intact, detached PbH2 structures.
#   Final X28/X29 structures must be independently classified.
#   Existing calculation directories are never overwritten.

from pathlib import Path
import csv
import json
import re

import numpy as np
from ase import Atoms
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SOURCE_ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR_source")
CONTROL = SOURCE_ROOT / "control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

RESULTS_CSV = ROOT / "analysis" / "surface_results.csv"

MANIFEST_DIR = SOURCE_ROOT / "campaigns"
MANIFEST_PATH = MANIFEST_DIR / "wave9_detached_PbH2_explicit_water_pilot.json"

OH_BOND_A = 0.970
HOH_ANGLE_DEG = 104.5
HYDRIDE_HH_CONTACT_A = 1.75

GO_MAXSTEP_A = 0.03
MIN_DETACHED_PB_LIFT_A = 4.0
MAX_PBH_A = 2.25
MIN_NON_H2_HH_A = 1.20


JOBS = [
    {
        "job_id": "X28",
        "source_job_id": "X06",
        "source_relpath": "surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_6p0",
        "source_target_mu_Ha": -0.11930,
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "job_name": "detH2O_mu1193",
        "relpath": "surfaces/Pb111/PbH2_extraction/explicit_water/detached/m0p1193",
        "balanced_surface_partner": "X21",
    },
    {
        "job_id": "X29",
        "source_job_id": "X14",
        "source_relpath": "surfaces/Pb111/PbH2_extraction/potential_sweep/detached/m0p09725",
        "source_target_mu_Ha": -0.09725,
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "job_name": "detH2O_mu09725",
        "relpath": "surfaces/Pb111/PbH2_extraction/explicit_water/detached/m0p09725",
        "balanced_surface_partner": "X22",
    },
]


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

def unit(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)

    if norm < 1e-10:
        raise ValueError("Cannot normalize near-zero vector.")

    return vector / norm


def perpendicular_component(vector, axis):
    vector = np.asarray(vector, dtype=float)
    axis = unit(axis)

    result = vector - np.dot(vector, axis) * axis

    if np.linalg.norm(result) > 1e-6:
        return unit(result)

    candidates = [
        np.array([1.0, 0.0, 0.0]),
        np.array([0.0, 1.0, 0.0]),
        np.array([0.0, 0.0, 1.0]),
    ]

    candidate = min(candidates, key=lambda v: abs(np.dot(v, axis)))
    return unit(candidate - np.dot(candidate, axis) * axis)


def angle_deg(v1, v2):
    cosine = np.clip(
        np.dot(unit(v1), unit(v2)),
        -1.0,
        1.0,
    )

    return float(np.degrees(np.arccos(cosine)))


def load_harvest():
    if not RESULTS_CSV.is_file():
        raise FileNotFoundError(
            f"Missing harvested results: {RESULTS_CSV}\n"
            "Run harvest_surface_results.py first."
        )

    rows = {}

    with RESULTS_CSV.open(newline="") as handle:
        for row in csv.DictReader(handle):
            rows[row["job_id"]] = row

    return rows


def surface_plane_z(atoms):
    symbols = np.array(atoms.get_chemical_symbols())
    pb_indices = np.where(symbols == "Pb")[0]

    if len(pb_indices) < 9:
        raise ValueError(f"Expected at least 9 Pb atoms; found {len(pb_indices)}.")

    pb_z = atoms.positions[pb_indices, 2]

    return float(
        np.median(
            np.sort(pb_z)[-9:]
        )
    )


# ---------------------------------------------------------------------
# Detached PbH2 parent validation
# ---------------------------------------------------------------------

def identify_detached_pbh2(atoms):
    atoms = atoms.copy()
    atoms.set_pbc((True, True, False))

    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(pb_indices) != 36:
        raise ValueError(
            f"Detached PbH2 parent should contain 36 Pb atoms; found {len(pb_indices)}."
        )

    if len(h_indices) != 2:
        raise ValueError(
            f"Detached PbH2 parent should contain exactly 2 H atoms; found {len(h_indices)}."
        )

    nearest_pb = []
    pb_h_distances = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(
                int(pb_idx),
                int(h_idx),
                mic=True,
            )
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))

        nearest_pb.append(
            int(pb_indices[local])
        )

        pb_h_distances.append(
            float(distances[local])
        )

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two H atoms do not share one Pb: {nearest_pb}"
        )

    if max(pb_h_distances) > MAX_PBH_A:
        raise ValueError(
            f"Parent does not look like intact PbH2: Pb-H = {pb_h_distances}"
        )

    hh = float(
        atoms.get_distance(
            int(h_indices[0]),
            int(h_indices[1]),
            mic=True,
        )
    )

    if hh < MIN_NON_H2_HH_A:
        raise ValueError(
            f"Parent looks H2-like rather than PbH2-like: H-H = {hh:.3f} Å"
        )

    pb_idx = nearest_pb[0]

    pb_lift = float(
        atoms.positions[pb_idx, 2]
        - surface_plane_z(atoms)
    )

    if pb_lift < MIN_DETACHED_PB_LIFT_A:
        raise ValueError(
            "Parent does not resemble detached PbH2 + vacancy: "
            f"Pb lift = {pb_lift:.3f} Å"
        )

    return {
        "Pb_index": pb_idx,
        "H_indices": [int(i) for i in h_indices],
        "PbH_distances_A": pb_h_distances,
        "HH_distance_A": hh,
        "Pb_lift_A": pb_lift,
    }


def validate_source_job(job, harvest):
    source_job_id = job["source_job_id"]
    source_dir = ROOT / job["source_relpath"]

    if source_job_id not in harvest:
        raise KeyError(
            f"{source_job_id} not found in {RESULTS_CSV}"
        )

    status = harvest[source_job_id]["status"]

    if status != "CONVERGED":
        raise RuntimeError(
            f"{source_job_id} is not converged: status={status}"
        )

    for filename in ["CONTCAR", "metadata.json"]:
        path = source_dir / filename

        if not path.is_file():
            raise FileNotFoundError(
                f"Missing required source file: {path}"
            )

    metadata = json.loads(
        (source_dir / "metadata.json").read_text()
    )

    if metadata.get("job_id") != source_job_id:
        raise ValueError(
            f"{source_dir}/metadata.json identifies "
            f"{metadata.get('job_id')}; expected {source_job_id}."
        )

    source_mu = metadata.get("target_mu_Ha")

    if source_mu is None or not np.isclose(
        float(source_mu),
        job["source_target_mu_Ha"],
        atol=1e-8,
        rtol=0.0,
    ):
        raise ValueError(
            f"{source_job_id} metadata target-mu = {source_mu}; "
            f"expected {job['source_target_mu_Ha']} Ha."
        )

    atoms = read(
        source_dir / "CONTCAR",
        format="vasp",
    )

    atoms.set_pbc(
        (True, True, False)
    )

    geometry = identify_detached_pbh2(atoms)

    return atoms, geometry


# ---------------------------------------------------------------------
# Explicit-water construction
# ---------------------------------------------------------------------

def local_geometry_vectors(atoms, pb_idx, h_indices, i):
    h_idx = h_indices[i]
    other_idx = h_indices[1 - i]

    pb_to_h = atoms.get_distance(
        pb_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    other_to_h = atoms.get_distance(
        other_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    outward = unit(pb_to_h)
    sideways = perpendicular_component(
        other_to_h,
        outward,
    )

    return outward, sideways


def make_hydride_oriented_water(
    target_position,
    outward,
    sideways,
):
    theta = np.deg2rad(
        HOH_ANGLE_DEG
    )

    donor_h = (
        target_position
        + HYDRIDE_HH_CONTACT_A * outward
    )

    oxygen = (
        donor_h
        + OH_BOND_A * outward
    )

    second_direction = (
        np.cos(theta) * (-outward)
        + np.sin(theta) * sideways
    )

    second_h = (
        oxygen
        + OH_BOND_A * unit(second_direction)
    )

    return Atoms(
        symbols=["O", "H", "H"],
        positions=[
            oxygen,
            donor_h,
            second_h,
        ],
    )


def add_two_waters(
    parent,
    pb_idx,
    h_indices,
):
    atoms = parent.copy()
    water_records = []

    for i, h_idx in enumerate(h_indices):
        outward, sideways = local_geometry_vectors(
            parent,
            pb_idx,
            h_indices,
            i,
        )

        target = parent.positions[h_idx].copy()

        water = make_hydride_oriented_water(
            target,
            outward,
            sideways,
        )

        water.set_cell(
            parent.cell
        )

        water.set_pbc(
            parent.pbc
        )

        first_new_index = len(atoms)

        atoms += water

        water_records.append(
            {
                "target_parent_H_index": int(h_idx),
                "water_initial_indices_before_go_sort": [
                    first_new_index,
                    first_new_index + 1,
                    first_new_index + 2,
                ],
            }
        )

    atoms.set_pbc(
        (True, True, False)
    )

    return atoms, water_records


def validate_geometry(
    atoms,
    pb_idx,
    hydride_indices,
    water_records,
):
    symbols = np.array(
        atoms.get_chemical_symbols()
    )

    n_pb = int(
        np.sum(symbols == "Pb")
    )

    n_o = int(
        np.sum(symbols == "O")
    )

    n_h = int(
        np.sum(symbols == "H")
    )

    if (n_pb, n_o, n_h) != (36, 2, 6):
        raise ValueError(
            f"Unexpected composition Pb/O/H = "
            f"{n_pb}/{n_o}/{n_h}; expected 36/2/6."
        )

    if len(water_records) != 2:
        raise ValueError(
            f"Expected two explicit-water records; found {len(water_records)}."
        )

    contacts = []
    water_geometry = []

    water_groups = [
        record["water_initial_indices_before_go_sort"]
        for record in water_records
    ]

    for i, group in enumerate(water_groups):
        o_idx, wh1_idx, wh2_idx = group
        target_h = hydride_indices[i]

        if (
            symbols[o_idx] != "O"
            or symbols[wh1_idx] != "H"
            or symbols[wh2_idx] != "H"
        ):
            raise ValueError(
                f"Unexpected explicit-water species ordering in water {i + 1}."
            )

        oh1 = float(
            atoms.get_distance(
                o_idx,
                wh1_idx,
                mic=True,
            )
        )

        oh2 = float(
            atoms.get_distance(
                o_idx,
                wh2_idx,
                mic=True,
            )
        )

        v1 = atoms.get_distance(
            o_idx,
            wh1_idx,
            mic=True,
            vector=True,
        )

        v2 = atoms.get_distance(
            o_idx,
            wh2_idx,
            mic=True,
            vector=True,
        )

        angle = angle_deg(
            v1,
            v2,
        )

        if not (
            0.94 <= oh1 <= 1.00
            and 0.94 <= oh2 <= 1.00
        ):
            raise ValueError(
                f"Water {i + 1} has bad initial O-H distances: "
                f"{oh1:.3f}, {oh2:.3f} Å"
            )

        if not 102.0 <= angle <= 107.0:
            raise ValueError(
                f"Water {i + 1} has bad initial H-O-H angle: "
                f"{angle:.2f} deg"
            )

        contact = float(
            min(
                atoms.get_distance(
                    target_h,
                    wh1_idx,
                    mic=True,
                ),
                atoms.get_distance(
                    target_h,
                    wh2_idx,
                    mic=True,
                ),
            )
        )

        if abs(
            contact
            - HYDRIDE_HH_CONTACT_A
        ) > 0.05:
            raise ValueError(
                f"Water {i + 1} target contact = {contact:.3f} Å; "
                f"expected approximately {HYDRIDE_HH_CONTACT_A:.3f} Å."
            )

        contacts.append(
            contact
        )

        water_geometry.append(
            {
                "OH_distances_A": [
                    oh1,
                    oh2,
                ],
                "HOH_angle_deg": angle,
                "hydride_contact_A": contact,
            }
        )

    interwater_distances = [
        atoms.get_distance(
            i,
            j,
            mic=True,
        )
        for i in water_groups[0]
        for j in water_groups[1]
    ]

    min_interwater = float(
        min(interwater_distances)
    )

    if min_interwater < 1.35:
        raise ValueError(
            "The two explicit waters overlap: "
            f"minimum cross-water distance = {min_interwater:.3f} Å"
        )

    pb_h = [
        float(
            atoms.get_distance(
                pb_idx,
                h_idx,
                mic=True,
            )
        )
        for h_idx in hydride_indices
    ]

    hh = float(
        atoms.get_distance(
            hydride_indices[0],
            hydride_indices[1],
            mic=True,
        )
    )

    max_z = float(
        np.max(
            atoms.positions[:, 2]
        )
    )

    top_gap = float(
        atoms.cell[2, 2]
        - max_z
    )

    if top_gap < 4.0:
        raise ValueError(
            f"Only {top_gap:.2f} Å remains above explicit atoms."
        )

    return {
        "PbH_distances_A": pb_h,
        "HH_distance_A": hh,
        "water_H_to_hydride_contacts_A": contacts,
        "water_geometry": water_geometry,
        "min_interwater_distance_A": min_interwater,
        "top_vacuum_gap_A": top_gap,
    }


# ---------------------------------------------------------------------
# JDFTx / Slurm construction
# ---------------------------------------------------------------------

def make_go_text(
    template,
    target_mu,
):
    text = template

    if re.search(
        r"(?m)^\s*target-mu\s+",
        text,
    ):
        raise RuntimeError(
            "Canonical go.py unexpectedly already contains target-mu."
        )

    fluid_marker = (
        "fluid-anion F- 0.5\n"
    )

    if text.count(
        fluid_marker
    ) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid-anion command in canonical go.py."
        )

    text = text.replace(
        fluid_marker,
        fluid_marker
        + f"target-mu {target_mu:.5f}\n",
        1,
    )

    if re.search(
        r"(?m)^dump Ionic .*Vcavity",
        text,
    ):
        raise RuntimeError(
            "Canonical go.py unexpectedly already requests Vcavity."
        )

    cavity_marker = (
        "dump Ionic BoundCharge VfluidTot FluidDensity\n"
    )

    if text.count(
        cavity_marker
    ) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid dump command in canonical go.py."
        )

    text = text.replace(
        cavity_marker,
        cavity_marker
        + "dump Ionic Vcavity\n",
        1,
    )

    text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={GO_MAXSTEP_A:.2f}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError(
            "Could not uniquely replace FIRE maxstep."
        )

    return text


def make_submit_text(
    template,
    job_name,
):
    text, n = re.subn(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        template,
        count=1,
    )

    if n != 1:
        raise RuntimeError(
            "Could not uniquely replace Slurm job name."
        )

    return text


# ---------------------------------------------------------------------
# Deterministic Git-tracked campaign definition
# ---------------------------------------------------------------------

manifest_jobs = [
    {
        "job_id": job["job_id"],
        "type": "geometry_optimization",
        "family": "PbH2_extraction_explicit_water",
        "directory": job["relpath"],
        "source_job_id": job["source_job_id"],
        "source_path": job["source_relpath"],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "initial_state": "detached_PbH2_plus_vacancy_plus_2H2O",
        "balanced_surface_partner": job["balanced_surface_partner"],
    }
    for job in JOBS
]

manifest = {
    "wave": 9,
    "name": "detached PbH2 explicit-water pilot",
    "manifest_role": "campaign_definition",
    "scientific_question": (
        "Can detached PbH2 + vacancy be stabilized with the same "
        "hydride-oriented two-water first-shell motif as the hydrated surface "
        "Pb(H)2 precursor, enabling balanced fixed-mu detachment thermodynamics?"
    ),
    "balanced_comparisons": [
        {
            "target_mu_Ha": -0.11930,
            "surface_job_id": "X21",
            "detached_job_id": "X28",
            "quantity": "G_X28 - G_X21",
        },
        {
            "target_mu_Ha": -0.09725,
            "surface_job_id": "X22",
            "detached_job_id": "X29",
            "quantity": "G_X29 - G_X22",
        },
    ],
    "potential_mapping_note": (
        "Approximate RHE potentials use the historical project calibration. "
        "target_mu_Ha is the authoritative calculation variable."
    ),
    "interpretation_note": (
        "Balanced G differences are physically meaningful only after both "
        "surface and detached hydrated states have converged to chemically "
        "physical structures."
    ),
    "jobs": manifest_jobs,
}


# ---------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------

for path in [
    GO_TEMPLATE,
    SUBMIT_TEMPLATE,
    RESULTS_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )

target_dirs = [
    ROOT / job["relpath"]
    for job in JOBS
]

existing = [
    path
    for path in target_dirs
    if path.exists()
]

if existing:
    raise FileExistsError(
        "Refusing to overwrite existing Wave 9 directories:\n"
        + "\n".join(
            str(path)
            for path in existing
        )
    )

if MANIFEST_PATH.exists():
    existing_manifest = json.loads(
        MANIFEST_PATH.read_text()
    )

    if existing_manifest != manifest:
        raise RuntimeError(
            f"Existing Git-tracked manifest differs from this Wave 9 definition: "
            f"{MANIFEST_PATH}\nRefusing to overwrite it."
        )

harvest = load_harvest()

parent_atoms = {}
parent_geometry = {}

for job in JOBS:
    atoms, geometry = validate_source_job(
        job,
        harvest,
    )

    parent_atoms[
        job["job_id"]
    ] = atoms

    parent_geometry[
        job["job_id"]
    ] = geometry


# Construct and validate all starting structures before creating directories.
starting_structures = {}
water_records_by_job = {}
starting_metrics = {}

for job in JOBS:
    parent = parent_atoms[
        job["job_id"]
    ]

    geometry = parent_geometry[
        job["job_id"]
    ]

    atoms, water_records = add_two_waters(
        parent,
        geometry["Pb_index"],
        geometry["H_indices"],
    )

    metrics = validate_geometry(
        atoms,
        geometry["Pb_index"],
        geometry["H_indices"],
        water_records,
    )

    starting_structures[
        job["job_id"]
    ] = atoms

    water_records_by_job[
        job["job_id"]
    ] = water_records

    starting_metrics[
        job["job_id"]
    ] = metrics


# Validate template transformations before writing jobs.
go_template = GO_TEMPLATE.read_text()
submit_template = SUBMIT_TEMPLATE.read_text()

go_texts = {}
submit_texts = {}

for job in JOBS:
    go_texts[
        job["job_id"]
    ] = make_go_text(
        go_template,
        job["target_mu_Ha"],
    )

    submit_texts[
        job["job_id"]
    ] = make_submit_text(
        submit_template,
        job["job_name"],
    )


# ---------------------------------------------------------------------
# Create X28-X29
# ---------------------------------------------------------------------

print(
    "Wave 9 detached PbH2 + explicit-water setup"
)
print(
    "==========================================="
)

for job in JOBS:
    job_id = job["job_id"]

    atoms = starting_structures[
        job_id
    ]

    metrics = starting_metrics[
        job_id
    ]

    source_geometry = parent_geometry[
        job_id
    ]

    target_dir = ROOT / job["relpath"]

    target_dir.mkdir(
        parents=True
    )

    write(
        target_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    write(
        target_dir / "initial.xyz",
        atoms,
        format="extxyz",
    )

    (
        target_dir / "go.py"
    ).write_text(
        go_texts[job_id]
    )

    (
        target_dir / "submit.sh"
    ).write_text(
        submit_texts[job_id]
    )

    metadata = {
        "job_id": job_id,
        "family": "PbH2_extraction_explicit_water",
        "initial_state": "detached_PbH2_plus_vacancy_plus_2H2O",
        "state_label_scope": "initial_intent",
        "purpose": (
            "Construct a detached PbH2 + vacancy product with two "
            "hydride-oriented explicit first-shell water molecules so that "
            "PbH2 detachment can be compared against the hydrated surface "
            "Pb(H)2 precursor at identical stoichiometry and target-mu."
        ),
        "source_job_id": job["source_job_id"],
        "source_path": job["source_relpath"],
        "source_geometry_validation": source_geometry,
        "balanced_surface_partner": job["balanced_surface_partner"],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "explicit_water_count": 2,
        "water_orientation": "hydride_oriented",
        "construction": {
            "OH_bond_A": OH_BOND_A,
            "HOH_angle_deg": HOH_ANGLE_DEG,
            "hydride_Hwater_contact_A": HYDRIDE_HH_CONTACT_A,
            "common_parent_Pb_index_before_go_sort": source_geometry["Pb_index"],
            "parent_H_indices_before_go_sort": source_geometry["H_indices"],
            "water_records_before_go_sort": water_records_by_job[job_id],
        },
        "initial_geometry_metrics": metrics,
        "kpts": [
            4,
            4,
            1,
            "gamma",
        ],
        "ase_pbc": [
            True,
            True,
            False,
        ],
        "n_atoms": len(atoms),
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": GO_MAXSTEP_A,
        "requested_fluid_diagnostics": [
            "FluidDensity",
            "BoundCharge",
            "VfluidTot",
            "Vcavity",
        ],
        "potential_mapping_note": (
            "approx_U_RHE_V_at_pH7 uses the historical project calibration; "
            "target_mu_Ha is the authoritative calculation variable."
        ),
    }

    (
        target_dir / "metadata.json"
    ).write_text(
        json.dumps(
            metadata,
            indent=2,
        )
        + "\n"
    )

    print()
    print(
        f"{job_id}  "
        f"mu={job['target_mu_Ha']:.5f}  "
        f"source={job['source_job_id']}  "
        f"partner={job['balanced_surface_partner']}"
    )

    print(
        f"source Pb lift:     "
        f"{source_geometry['Pb_lift_A']:.4f} Å"
    )

    print(
        "Pb-H:              "
        + ", ".join(
            f"{x:.4f} Å"
            for x in metrics["PbH_distances_A"]
        )
    )

    print(
        f"H-H:               "
        f"{metrics['HH_distance_A']:.4f} Å"
    )

    print(
        "water contacts:    "
        + ", ".join(
            f"{x:.4f} Å"
            for x in metrics["water_H_to_hydride_contacts_A"]
        )
    )

    print(
        f"min water-water:   "
        f"{metrics['min_interwater_distance_A']:.4f} Å"
    )

    print(
        f"top vacuum gap:     "
        f"{metrics['top_vacuum_gap_A']:.4f} Å"
    )


# ---------------------------------------------------------------------
# Write / validate Git-tracked Wave 9 manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if MANIFEST_PATH.exists():
    print()
    print(
        f"Manifest already exists and matches: "
        f"{MANIFEST_PATH}"
    )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2,
        )
        + "\n"
    )

    print()
    print(
        f"Wrote manifest: "
        f"{MANIFEST_PATH}"
    )